In [ ]:
!pip install -q gradio ultralytics opencv-python-headless
!apt-get install -q ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.2 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [ ]:


import os
import sys
import json
import math
import time
import shutil
import hashlib
import base64
import warnings
import subprocess
import traceback
from collections import deque, defaultdict, Counter

warnings.filterwarnings("ignore")

def _mount_drive_if_colab():
    if os.path.isdir("/content"):
        try:
            from google.colab import drive
            if not os.path.isdir("/content/drive/MyDrive"):
                print("Connecting to Google Drive...")
                drive.mount("/content/drive")
        except Exception as e:
            print(f"Warning: Could not mount Google Drive: {e}")

_mount_drive_if_colab()

# ------------------------------------------------------------------ dependencies
def _pip_install(*pkgs):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, timeout=1800)
    except Exception:
        pass

def _install_deps():
    if os.environ.get("LG_SKIP_INSTALL", "0") == "1":
        return
    _pip_install("gradio", "ultralytics", "opencv-python-headless",
                 "pandas", "openpyxl", "Pillow")
    try:
        subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=900)
    except Exception:
        pass

_install_deps()

import cv2
import numpy as np
import pandas as pd

try:
    import gradio as gr
    GRADIO_OK = True
except Exception:
    GRADIO_OK = False
    class _GrStub:
        class SelectData:
            pass
        class Progress:
            def __call__(self, *a, **k):
                return None
        @staticmethod
        def update(*a, **k):
            return None
    gr = _GrStub()

_MODEL = None
_MODEL_INFO = {"ok": False, "msg": "Model not loaded yet.", "path": None}
_DEVICE = None

# ------------------------------------------------------------------ paths
MODEL_PATH = os.environ.get(
    "LG_MODEL_PATH",
    "/content/drive/MyDrive/theproject/best.pt")

# Set save directory directly to Google Drive if in Colab
if os.path.isdir("/content"):
    DEFAULT_SAVE_DIR = "/content/drive/MyDrive/Lane_Guardian_Data"
else:
    DEFAULT_SAVE_DIR = "./atms_data"

SAVE_DIR = os.environ.get("LG_SAVE_DIR", DEFAULT_SAVE_DIR)

CALIB_DIR = os.path.join(SAVE_DIR, "calibrations")
EVID_DIR = os.path.join(SAVE_DIR, "evidence")
for _d in (SAVE_DIR, CALIB_DIR, EVID_DIR):
    os.makedirs(_d, exist_ok=True)

VIDEO_CACHE = os.path.join(SAVE_DIR, "session_video.mp4")
BASE_FRAME = os.path.join(SAVE_DIR, "base_frame.jpg")
BG_FRAME = os.path.join(SAVE_DIR, "background.jpg")
DIAG_IMAGE = os.path.join(SAVE_DIR, "calibration_diagnostics.jpg")
TRACKER_CFG = os.path.join(SAVE_DIR, "lg_bytetrack.yaml")
OUT_VIDEO = os.path.join(SAVE_DIR, "lane_guardian_output.mp4")
XLSX_PATH = os.path.join(SAVE_DIR, "traffic_report.xlsx")

APP_NAME = "Lane Guardian"
APP_VER = "V30 PRO"
CALIB_SCHEMA = 3

# --------------------------------------------------------- vehicle taxonomy
PCU = {
    "truck": 3.0, "lorry": 3.0, "hgv": 3.0, "trailer": 3.5,
    "bus": 3.0, "van": 1.5, "lcv": 1.5, "tempo": 1.5,
    "car": 1.0, "jeep": 1.0, "suv": 1.0, "taxi": 1.0,
    "auto": 0.8, "rickshaw": 0.8, "tuktuk": 0.8, "three": 0.8,
    "motorcycle": 0.5, "motorbike": 0.5, "motor": 0.5, "moto": 0.5,
    "bike": 0.5, "scooter": 0.5, "scooty": 0.5, "cycle": 0.5, "bicycle": 0.5,
    "tractor": 1.5, "cart": 2.0,
}

VEH_WIDTH_M = {
    "truck": 2.5, "lorry": 2.5, "trailer": 2.5, "bus": 2.5,
    "van": 1.9, "lcv": 1.9, "tempo": 1.8,
    "car": 1.75, "suv": 1.85, "jeep": 1.8, "taxi": 1.75,
    "auto": 1.4, "rickshaw": 1.4, "tuktuk": 1.4,
    "motorcycle": 0.75, "motorbike": 0.75, "motor": 0.75, "moto": 0.75,
    "bike": 0.75, "scooter": 0.7, "scooty": 0.7, "cycle": 0.6, "bicycle": 0.6,
    "tractor": 2.0,
}
VEH_LEN_M = {
    "truck": 8.0, "lorry": 8.0, "trailer": 14.0, "bus": 11.0,
    "van": 5.0, "lcv": 5.0, "tempo": 4.5,
    "car": 4.2, "suv": 4.6, "jeep": 4.4, "taxi": 4.2,
    "auto": 2.8, "rickshaw": 2.8, "tuktuk": 2.8,
    "motorcycle": 2.0, "motorbike": 2.0, "motor": 2.0, "moto": 2.0,
    "bike": 2.0, "scooter": 1.8, "scooty": 1.8, "cycle": 1.7, "bicycle": 1.7,
}

VEHICLE_KEYWORDS = tuple(set(list(PCU.keys()) + ["vehicle", "hmv", "mcv", "e-rickshaw"]))
NON_VEHICLE_KEYWORDS = ("person", "pedestrian", "traffic", "light", "sign", "hydrant",
                        "bench", "bird", "dog", "cat", "cow", "animal", "umbrella",
                        "backpack", "handbag", "pole", "tree", "train", "boat",
                        "airplane", "kite", "bottle", "chair")

CLS_COLORS = {
    "car": (0, 165, 255), "suv": (0, 165, 255), "taxi": (0, 200, 255),
    "auto": (0, 255, 255), "rickshaw": (0, 255, 255),
    "bike": (0, 255, 0), "motor": (0, 255, 0), "moto": (0, 255, 0),
    "scooter": (0, 255, 0), "cycle": (0, 255, 0), "bicycle": (0, 255, 0),
    "bus": (255, 0, 0), "truck": (255, 0, 255), "lorry": (255, 0, 255),
    "van": (255, 128, 0), "tractor": (128, 128, 255),
}
DEFAULT_COLOR = (200, 200, 200)

# ------------------------------------------------------------------ violations
VIOLATIONS = {
    "Wrong-Way":                 ("critical", 30.0, False),
    "Overspeeding":              ("critical", 15.0, True),
    "Lane Straddling":           ("high",     8.0,  True),
    "Unsafe Lane Change":        ("high",     6.0,  True),
    "Unauthorized Parking":      ("medium",   60.0, False),
    "Illegal Lane Change":       ("high",     8.0,  True),
}
SEV_LABELS = {"critical": "Critical", "high": "High", "medium": "Medium", "low": "Low"}
SEV_COLORS = {"critical": "#ef4444", "high": "#f97316", "medium": "#eab308", "low": "#6b7280"}
SEV_ORDER = {"critical": 0, "high": 1, "medium": 2, "low": 3}

VIOL_BOX_COLOR = {
    "Wrong-Way": (0, 0, 220),
    "Overspeeding": (0, 100, 255),
    "Lane Straddling": (0, 0, 255),
    "Illegal Lane Change": (40, 40, 255),
    "Unsafe Lane Change": (0, 90, 255),
    "Unauthorized Parking": (0, 180, 255),
}

# ------------------------------------------------------------------ tunables
CAL_SAMPLES = 48
CAL_MAX_W = 1280
PRESCAN_MAX_FRAMES = 320
PRESCAN_TARGET_FPS = 6.0
BEV_W, BEV_H = 640, 900
MIN_LANE_W_M, MAX_LANE_W_M = 2.4, 5.5
DEFAULT_LANE_WIDTH_M = 3.5
DEFAULT_SPEED_LIMIT = 80
DEFAULT_STOP_SEC = 5.0
FOV_FALLBACK_FACTOR = 0.90
F_PX_MIN_FACTOR, F_PX_MAX_FACTOR = 0.35, 4.00

MARKING_STANDARDS = {
    "Auto (measure & guess)":          (None, "let the calibrator infer")
}

STRADDLE_OVERLAP = 0.28
STRADDLE_SEC = 1.8
LANE_CHANGE_SETTLE_SEC = 0.55
LANE_CHANGE_FAST_SEC = 0.85
WRONGWAY_MIN_DISP_M = 3.0
WRONGWAY_MIN_SEC = 1.0
WRONGWAY_MIN_SUPPORT = 6
STOP_DISP_M = 1.2
SPEED_WIN_SEC = 0.7
MIN_TRACK_SEC = 0.35

ENGINE_MODES = [
    "Analyze by trace + trajectories (best)",
    "Analyze by trace only (fast, needs markings)",
    "Trajectories only (unmarked roads)",
]

# ------------------------------------------------------------------ tiny helpers
def _ts(sec):
    sec = max(0.0, float(sec))
    m, s = divmod(int(round(sec)), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def _kw_lookup(name, table, default):
    n = str(name).lower().replace("_", " ").replace("-", " ")
    best, best_len = default, 0
    for k, v in table.items():
        if k in n and len(k) > best_len:
            best, best_len = v, len(k)
    return best

def _pcu(name, with_match=False):
    n = str(name).lower()
    best, blen = 1.0, 0
    for k, v in PCU.items():
        if k in n and len(k) > blen:
            best, blen = v, len(k)
    return (best, blen > 0) if with_match else best

def _veh_width_m(name):
    return _kw_lookup(name, VEH_WIDTH_M, 1.8)

def _veh_len_m(name):
    return _kw_lookup(name, VEH_LEN_M, 4.2)

def _cls_color(name):
    n = str(name).lower()
    best, blen = DEFAULT_COLOR, 0
    for k, c in CLS_COLORS.items():
        if k in n and len(k) > blen:
            best, blen = c, len(k)
    return best

def _is_2w(name):
    n = str(name).lower()
    return any(w in n for w in ("motor", "moto", "bike", "scooter", "scooty", "cycle"))

def _is_vehicle(name):
    n = str(name).lower().replace("_", " ")
    if any(w in n for w in NON_VEHICLE_KEYWORDS):
        return False
    return any(w in n for w in VEHICLE_KEYWORDS)

def _clamp(v, lo, hi):
    return lo if v < lo else (hi if v > hi else v)

def _safe_div(a, b, d=0.0):
    return a / b if b not in (0, 0.0) else d

def _mad(v):
    v = np.asarray(v, dtype=np.float32).ravel()
    if v.size < 2:
        return 0.0
    m = float(np.median(v))
    return 1.4826 * float(np.median(np.abs(v - m)))

def _med_mad_hi(v, floor, n_sigma=3.0):
    v = np.asarray(v, dtype=np.float32).ravel()
    if v.size < 2:
        return float(floor)
    return float(np.median(v)) + max(float(floor), n_sigma * _mad(v))

def _smooth1d(y, sigma):
    y = np.asarray(y, dtype=np.float64)
    if sigma <= 0 or y.size < 3:
        return y
    rad = max(1, int(round(3 * sigma)))
    x = np.arange(-rad, rad + 1, dtype=np.float64)
    k = np.exp(-0.5 * (x / sigma) ** 2)
    k /= k.sum()
    pad = np.r_[np.full(rad, y[0]), y, np.full(rad, y[-1])]
    return np.convolve(pad, k, mode="valid")

def _find_peaks(y, min_dist, min_height=None, max_n=64):
    y = np.asarray(y, dtype=np.float64)
    n = y.size
    if n < 3:
        return []
    if min_height is None:
        min_height = y.mean() + 0.25 * y.std()
    cand = [i for i in range(1, n - 1)
            if y[i] >= y[i - 1] and y[i] >= y[i + 1] and y[i] >= min_height]
    if y[0] >= min_height and y[0] > y[1]:
        cand.insert(0, 0)
    if y[-1] >= min_height and y[-1] > y[-2]:
        cand.append(n - 1)
    cand.sort(key=lambda i: -y[i])
    keep = []
    for i in cand:
        if all(abs(i - j) >= min_dist for j in keep):
            keep.append(i)
        if len(keep) >= max_n:
            break
    return sorted(keep)

def _dominant_period(sig, lo, hi):
    y = np.asarray(sig, dtype=np.float64)
    if y.size < 8:
        return None, 0.0
    y = y - y.mean()
    if not np.any(y):
        return None, 0.0
    ac = np.correlate(y, y, mode="full")[y.size - 1:]
    if ac[0] <= 0:
        return None, 0.0
    ac = ac / ac[0]
    lo = int(max(2, lo))
    hi = int(min(len(ac) - 2, hi))
    if hi <= lo + 1:
        return None, 0.0
    seg = ac[lo:hi]
    k = int(np.argmax(seg))
    idx = lo + k
    strength = float(_clamp(seg[k], 0.0, 1.0))
    if 0 < idx < len(ac) - 1:
        a, b, c = ac[idx - 1], ac[idx], ac[idx + 1]
        den = (a - 2 * b + c)
        if abs(den) > 1e-9:
            idx = idx + 0.5 * (a - c) / den
    return float(idx), strength

def _line_from_seg(x1, y1, x2, y2):
    p1 = np.array([x1, y1, 1.0])
    p2 = np.array([x2, y2, 1.0])
    return np.cross(p1, p2)

def _file_sig(path, nbytes=1 << 20):
    try:
        sz = os.path.getsize(path)
        h = hashlib.md5(str(sz).encode())
        with open(path, "rb") as f:
            h.update(f.read(nbytes))
            if sz > 2 * nbytes:
                f.seek(-nbytes, os.SEEK_END)
                h.update(f.read(nbytes))
        return h.hexdigest()[:16]
    except Exception:
        return "unknown"

def _label(frame, text, x, y, color, scale=0.5, thick=1):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), bl = cv2.getTextSize(text, font, scale, thick)
    h, w = frame.shape[:2]
    x = int(_clamp(x, 0, max(0, w - tw - 8)))
    ty = int(_clamp(y, th + 8, h - 2))
    cv2.rectangle(frame, (x, ty - th - 7), (x + tw + 7, ty + bl - 2), color, -1)
    bright = 0.114 * color[0] + 0.587 * color[1] + 0.299 * color[2]
    cv2.putText(frame, text, (x + 4, ty - 3), font, scale,
                (0, 0, 0) if bright > 140 else (255, 255, 255), thick, cv2.LINE_AA)

# ==============================================================================
#  SECTION 1 - MODEL + VIDEO PLUMBING
# ==============================================================================
def _torch():
    import torch
    return torch

def get_device():
    global _DEVICE
    if _DEVICE is None:
        try:
            t = _torch()
            _DEVICE = 0 if t.cuda.is_available() else "cpu"
        except Exception:
            _DEVICE = "cpu"
    return _DEVICE

def device_label():
    d = get_device()
    if d == 0:
        try:
            return f"GPU ({_torch().cuda.get_device_name(0)})"
        except Exception:
            return "GPU"
    return "CPU"

def get_model(path=None):
    global _MODEL, _MODEL_INFO
    want = path or MODEL_PATH
    if _MODEL is not None and _MODEL_INFO.get("path") in (want, "yolov10n.pt"):
        return _MODEL, _MODEL_INFO
    try:
        from ultralytics import YOLO
    except Exception as e:
        _MODEL_INFO = {"ok": False, "path": None,
                       "msg": f"ultralytics is not installed ({e})."}
        return None, _MODEL_INFO
    if want and os.path.exists(want):
        try:
            _MODEL = YOLO(want)
            _MODEL_INFO = {"ok": True, "path": want, "custom": True,
                           "msg": f"Custom model loaded: {os.path.basename(want)}"}
            return _MODEL, _MODEL_INFO
        except Exception as e:
            fallback_msg = f"Custom weights failed to load ({e}); using yolov10n."
    else:
        fallback_msg = ("Custom weights not found; using COCO yolov10n "
                        "(non-vehicle classes are filtered out).")
    try:
        _MODEL = YOLO("yolov10n.pt")
        _MODEL_INFO = {"ok": True, "path": "yolov10n.pt", "custom": False,
                       "msg": fallback_msg}
    except Exception as e:
        _MODEL, _MODEL_INFO = None, {"ok": False, "path": None,
                                     "msg": f"No usable model ({e})."}
    return _MODEL, _MODEL_INFO

def ensure_tracker_cfg():
    if not os.path.exists(TRACKER_CFG):
        with open(TRACKER_CFG, "w") as f:
            f.write(
                "tracker_type: bytetrack\n"
                "track_high_thresh: 0.45\n"
                "track_low_thresh: 0.10\n"
                "new_track_thresh: 0.50\n"
                "track_buffer: 60\n"
                "match_thresh: 0.85\n"
                "fuse_score: True\n")
    return TRACKER_CFG

def _fourcc(code):
    for fn in (getattr(cv2, "VideoWriter_fourcc", None),
               getattr(getattr(cv2, "VideoWriter", None), "fourcc", None)):
        if callable(fn):
            try:
                return int(fn(*code))
            except Exception:
                continue
    return 0x7634706D          # 'mp4v' little-endian, hard fallback

def video_meta(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    if fps <= 1 or fps > 240:
        fps = 25.0
    return {"ok": (w > 0 and h > 0), "fps": float(fps), "w": w, "h": h, "frames": n,
            "dur": (n / fps if n > 0 else 0.0)}

def sample_frames(path, n=CAL_SAMPLES, max_w=CAL_MAX_W):
    meta = video_meta(path)
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return [], 1.0, meta
    total = meta["frames"]
    frames = []
    scale = 1.0
    if meta["w"] > 0:
        scale = min(1.0, float(max_w) / float(meta["w"]))

    def _prep(fr):
        if scale < 0.999:
            return cv2.resize(fr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        return fr

    ok_seek = False
    if total > n * 2:
        idxs = np.linspace(total * 0.02, total * 0.98, n).astype(int)
        got = 0
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ok, fr = cap.read()
            if ok and fr is not None:
                frames.append(_prep(fr))
                got += 1
        ok_seek = got >= max(6, n // 3)

    if not ok_seek:
        frames = []
        cap.release()
        cap = cv2.VideoCapture(path)
        step = max(1, int(total / n)) if total > 0 else max(1, int(meta["fps"] // 2))
        i = 0
        while len(frames) < n:
            ok, fr = cap.read()
            if not ok or fr is None:
                break
            if i % step == 0:
                frames.append(_prep(fr))
            i += 1
            if i > 200000:
                break
    cap.release()
    return frames, scale, meta


# ==============================================================================
#  SECTION 2 - SCENE UNDERSTANDING
# ==============================================================================
def temporal_stats(frames):
    if not frames:
        return None, None
    stack = np.stack(frames, axis=0)
    bg = np.median(stack, axis=0).astype(np.uint8)
    grays = np.stack([cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames],
                     axis=0).astype(np.float32)
    act = grays.std(axis=0)
    return bg, act

def activity_mask(act, frame_shape):
    if act is None:
        return None
    h, w = frame_shape[:2]
    a = act.copy()
    a = cv2.GaussianBlur(a, (0, 0), max(1.0, w / 300.0))
    mx = float(a.max())
    if mx < 2.0:
        return np.zeros((h, w), np.uint8)
    u8 = np.clip(a / mx * 255.0, 0, 255).astype(np.uint8)
    thr, m = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    floor = max(6.0, 0.10 * mx)
    m = ((a > floor) & (m > 0)).astype(np.uint8) * 255
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                  (max(3, int(w * 0.012)) | 1, max(3, int(h * 0.012)) | 1))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=2)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN,
                         cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    return m

def _rowwise_fill(mask, min_w_frac=0.02):
    h, w = mask.shape[:2]
    out = np.zeros_like(mask)
    for y in range(h):
        xs = np.flatnonzero(mask[y])
        if xs.size == 0:
            continue
        x0, x1 = int(xs[0]), int(xs[-1])
        if (x1 - x0) < min_w_frac * w:
            continue
        out[y, x0:x1 + 1] = 255
    return out

def road_corridor(bg, act_mask):
    h, w = bg.shape[:2]
    hsv = cv2.cvtColor(bg, cv2.COLOR_BGR2HSV)
    S, V = hsv[..., 1].astype(np.float32), hsv[..., 2].astype(np.float32)
    gray = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY)
    lap = np.abs(cv2.Laplacian(gray, cv2.CV_32F, ksize=3))
    tex = cv2.GaussianBlur(lap, (0, 0), max(1.5, w / 250.0))

    seed = np.zeros((h, w), np.uint8)
    if act_mask is not None and int(np.count_nonzero(act_mask)) > 200:
        er = max(3, int(min(h, w) * 0.010)) | 1
        seed = cv2.erode((act_mask > 0).astype(np.uint8),
                         cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (er, er)))
        if int(np.count_nonzero(seed)) < 200:
            seed = (act_mask > 0).astype(np.uint8)
    else:
        seed[int(h * 0.72):, int(w * 0.25):int(w * 0.75)] = 1
    sm = seed > 0
    if int(sm.sum()) > 200:
        s_thr = _med_mad_hi(S[sm], 18.0, 3.0)
        t_thr = _med_mad_hi(tex[sm], 5.0, 3.0)
        v_med = float(np.median(V[sm]))
        v_pad = max(45.0, 4.0 * _mad(V[sm]))
        v_lo, v_hi = max(10.0, v_med - v_pad), min(252.0, v_med + v_pad)
    else:
        s_thr = max(60.0, float(np.percentile(S, 45)))
        t_thr = max(2.0, float(np.percentile(tex, 62)))
        v_lo, v_hi = 22.0, 248.0
    cand = ((S < s_thr) & (tex < t_thr) & (V > v_lo) & (V < v_hi)).astype(np.uint8) * 255
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                  (max(3, int(w * 0.02)) | 1, max(3, int(h * 0.02)) | 1))
    cand = cv2.morphologyEx(cand, cv2.MORPH_CLOSE, k, iterations=2)

    ncomp, labels = cv2.connectedComponents(cand)
    best_lab, best_ov = 0, 0
    for lab in range(1, ncomp):
        ov = int(np.count_nonzero((labels == lab) & (seed > 0)))
        if ov > best_ov:
            best_lab, best_ov = lab, ov
    if best_lab == 0:
        sizes = [(int(np.count_nonzero(labels == l)), l) for l in range(1, ncomp)]
        best_lab = max(sizes)[1] if sizes else 0

    road = (labels == best_lab).astype(np.uint8) * 255 if best_lab else np.zeros((h, w), np.uint8)
    if act_mask is not None:
        road = cv2.bitwise_or(road, act_mask)

    band = _rowwise_fill(road)
    rows_hit = int(np.count_nonzero(band.any(axis=1)))
    widths = np.count_nonzero(band, axis=1)
    widths = widths[widths > 0]
    w_med = float(np.median(widths)) if widths.size else 0.0
    cover = float(np.count_nonzero(band)) / float(h * w)
    if rows_hit < 0.12 * h or w_med < 0.035 * w or cover > 0.93:
        band = np.zeros((h, w), np.uint8)
        band[int(h * 0.33):, :] = 255
    return band

def corridor_extent(band):
    h, w = band.shape[:2]
    left = np.full(h, -1, np.int32)
    right = np.full(h, -1, np.int32)
    for y in range(h):
        xs = np.flatnonzero(band[y])
        if xs.size:
            left[y], right[y] = int(xs[0]), int(xs[-1])
    rows = np.flatnonzero(left >= 0)
    if rows.size == 0:
        return left, right, int(h * 0.35), h - 1
    return left, right, int(rows[0]), int(rows[-1])

def paint_mask(bg, band):
    h, w = bg.shape[:2]
    lab = cv2.cvtColor(bg, cv2.COLOR_BGR2LAB)
    L = lab[..., 0]
    A = lab[..., 1].astype(np.int16) - 128
    B = lab[..., 2].astype(np.int16) - 128

    resp = np.zeros((h, w), np.float32)
    for frac in (0.006, 0.011, 0.020, 0.034):
        kw = max(3, int(round(w * frac))) | 1
        ker = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, 1))
        th = cv2.morphologyEx(L, cv2.MORPH_TOPHAT, ker)
        resp = np.maximum(resp, th.astype(np.float32))
    ks = max(3, int(round(min(h, w) * 0.012))) | 1
    resp = np.maximum(resp, 0.85 * cv2.morphologyEx(
        L, cv2.MORPH_TOPHAT,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ks, ks))).astype(np.float32))

    inside = band > 0
    if not inside.any():
        inside = np.ones((h, w), bool)
    vals = resp[inside]
    thr = max(6.0, float(np.mean(vals) + 1.9 * np.std(vals)),
              float(np.percentile(vals, 96)))

    Lf = L.astype(np.float32)
    road_L = float(np.median(Lf[inside]))
    brighter = Lf > road_L + max(8.0, 0.35 * float(np.std(Lf[inside])))
    white = (np.abs(A) < 16) & (np.abs(B) < 20)
    yellow = (B > 16) & (A > -12)
    chroma_ok = white | yellow

    m = ((resp > thr) & brighter & chroma_ok & inside).astype(np.uint8) * 255

    ncomp, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    keep = np.zeros((h, w), np.uint8)
    max_bw = max(4.0, w * 0.075)
    min_area = max(4.0, (w * h) * 2.0e-6)
    for i in range(1, ncomp):
        x, y, bw, bh, area = stats[i]
        if area < min_area:
            continue
        if bw > max_bw and bw > 2.2 * bh:
            continue
        if bw > max_bw * 2.0:
            continue
        keep[labels == i] = 255
    if np.count_nonzero(keep) < 0.15 * max(1, np.count_nonzero(m)):
        keep = m
    return keep

# ==============================================================================
#  SECTION 3 - VANISHING POINT + BIRD'S-EYE HOMOGRAPHY
# ==============================================================================
def segments_from_mask(mask, frame_shape, min_len_frac=0.025):
    h, w = frame_shape[:2]
    segs = []
    try:
        lines = cv2.HoughLinesP(
            mask, 1, np.pi / 360,
            threshold=max(12, int(h * 0.018)),
            minLineLength=max(8, int(h * min_len_frac)),
            maxLineGap=max(4, int(h * 0.02)))
    except Exception:
        lines = None
    if lines is None:
        return segs
    for l in lines:
        x1, y1, x2, y2 = [float(v) for v in np.ravel(l)[:4]]
        dx, dy = x2 - x1, y2 - y1
        ln = math.hypot(dx, dy)
        if ln < h * min_len_frac:
            continue
        ang = abs(math.degrees(math.atan2(dy, dx)))
        if ang < 12 or ang > 168:
            continue
        segs.append((x1, y1, x2, y2, ln))
    return segs

def ransac_vanishing_point(segs, frame_shape, road_top, iters=600, seed=12345):
    h, w = frame_shape[:2]
    if len(segs) < 2:
        return None, 0.0, 0
    rng = np.random.default_rng(seed)
    lines = np.array([_line_from_seg(*s[:4]) for s in segs], dtype=np.float64)
    wts = np.array([s[4] for s in segs], dtype=np.float64)
    angs = np.array([math.atan2(s[3] - s[1], s[2] - s[0]) for s in segs])
    tau = 0.010 * math.hypot(w, h)
    n = len(segs)
    y_hi = road_top + 0.10 * h
    best = (None, -1.0, 0)
    for _ in range(iters):
        i, j = rng.integers(0, n, 2)
        if i == j:
            continue
        if abs(math.sin(angs[i] - angs[j])) < 0.03:
            continue
        p = np.cross(lines[i], lines[j])
        if abs(p[2]) < 1e-8:
            continue
        vx, vy = p[0] / p[2], p[1] / p[2]
        if not (np.isfinite(vx) and np.isfinite(vy)):
            continue
        if vx < -2.5 * w or vx > 3.5 * w or vy < -4.0 * h or vy > y_hi:
            continue
        d = np.abs(lines[:, 0] * vx + lines[:, 1] * vy + lines[:, 2]) / \
            (np.hypot(lines[:, 0], lines[:, 1]) + 1e-9)
        inl = d < tau
        score = float(wts[inl].sum())
        if score > best[1]:
            best = ((vx, vy), score, int(inl.sum()))
    if best[0] is None:
        return None, 0.0, 0

    vp, score, _ = best
    for _ in range(3):
        d = np.abs(lines[:, 0] * vp[0] + lines[:, 1] * vp[1] + lines[:, 2]) / \
            (np.hypot(lines[:, 0], lines[:, 1]) + 1e-9)
        inl = d < tau * 1.5
        if inl.sum() < 2:
            break
        Ln = lines[inl] / (np.hypot(lines[inl, 0], lines[inl, 1])[:, None] + 1e-9)
        Wt = np.sqrt(wts[inl])[:, None]
        Amat = Ln[:, :2] * Wt
        bvec = -Ln[:, 2:3] * Wt
        try:
            sol, *_ = np.linalg.lstsq(Amat, bvec, rcond=None)
            nvx, nvy = float(sol[0, 0]), float(sol[1, 0])
        except Exception:
            break
        if not (np.isfinite(nvx) and np.isfinite(nvy)):
            break
        if nvx < -2.5 * w or nvx > 3.5 * w or nvy < -4.0 * h or nvy > y_hi:
            break
        vp = (nvx, nvy)
    d = np.abs(lines[:, 0] * vp[0] + lines[:, 1] * vp[1] + lines[:, 2]) / \
        (np.hypot(lines[:, 0], lines[:, 1]) + 1e-9)
    inl = int((d < tau * 1.5).sum())
    conf = _clamp(inl / max(6.0, 0.35 * n), 0.0, 1.0)
    return vp, conf, inl

def vp_from_tracks(tracks, frame_shape, min_pts=8):
    h, w = frame_shape[:2]
    segs = []
    for pts in tracks.values():
        if len(pts) < min_pts:
            continue
        P = np.asarray(pts, dtype=np.float64)[:, :2]
        disp = math.hypot(P[-1, 0] - P[0, 0], P[-1, 1] - P[0, 1])
        if disp < 0.05 * h:
            continue
        c = P.mean(axis=0)
        u, s, vt = np.linalg.svd(P - c, full_matrices=False)
        d = vt[0]
        if abs(d[1]) < 1e-6:
            continue
        p1 = c - d * disp
        p2 = c + d * disp
        segs.append((p1[0], p1[1], p2[0], p2[1], disp))
    if len(segs) < 2:
        return None, 0.0, 0
    return ransac_vanishing_point(segs, frame_shape, road_top=0.0, iters=500, seed=777)

def fuse_vps(vp_paint, c_paint, vp_track, c_track, frame_shape):
    h, w = frame_shape[:2]
    if vp_paint is None and vp_track is None:
        return None, 0.0, "none"
    if vp_paint is None:
        return vp_track, c_track, "trajectories"
    if vp_track is None:
        return vp_paint, c_paint, "markings"
    dx = abs(vp_paint[0] - vp_track[0]) / max(1.0, w)
    dy = abs(vp_paint[1] - vp_track[1]) / max(1.0, h)
    if dx < 0.18 and dy < 0.25:
        wp, wt = max(0.05, c_paint), max(0.05, c_track) * 1.35
        tot = wp + wt
        vp = ((vp_paint[0] * wp + vp_track[0] * wt) / tot,
              (vp_paint[1] * wp + vp_track[1] * wt) / tot)
        return vp, _clamp(0.55 + 0.45 * min(c_paint, c_track), 0, 1), "markings+trajectories"
    if c_track * 1.25 >= c_paint:
        return vp_track, c_track * 0.85, "trajectories (markings disagreed)"
    return vp_paint, c_paint * 0.85, "markings (trajectories disagreed)"

def _project_row(vp, x, y, y_target):
    den = (y - vp[1])
    if abs(den) < 1e-6:
        return x
    t = (y_target - vp[1]) / den
    return vp[0] + t * (x - vp[0])

def _corridor_span_at(left, right, r_top, y_near, vp, d_near, frame_w,
                      max_gain=1.6):
    y_i = int(y_near)
    xl0, xr0 = float(left[y_i]), float(right[y_i])
    y_lim = max(r_top, vp[1] + d_near / max(1.05, max_gain))
    ys = np.arange(int(math.ceil(y_lim)), y_i + 1)
    ys = ys[(ys >= 0) & (ys < left.shape[0])]
    ys = ys[(left[ys] >= 0) & (right[ys] > left[ys])]
    if ys.size < 8:
        return xl0, xr0
    t = (y_near - vp[1]) / (ys.astype(np.float64) - vp[1])
    pl = vp[0] + t * (left[ys] - vp[0])
    pr = vp[0] + t * (right[ys] - vp[0])
    xl = float(np.percentile(pl, 12))
    xr = float(np.percentile(pr, 88))
    w_med = float(np.median(pr - pl))
    if not (np.isfinite(xl) and np.isfinite(xr)) or xr - xl < 8:
        return xl0, xr0
    if xr - xl > 1.5 * w_med:
        c = 0.5 * (float(np.median(pl)) + float(np.median(pr)))
        xl, xr = c - 0.75 * w_med, c + 0.75 * w_med
    xl, xr = min(xl, xl0), max(xr, xr0)
    return (_clamp(xl, -0.20 * frame_w, 0.98 * frame_w),
            _clamp(xr, 0.02 * frame_w, 1.20 * frame_w))

def build_ipm(vp, band, frame_shape, depth_ratio=0.32, lateral_pad=0.07):
    h, w = frame_shape[:2]
    left, right, r_top, r_bot = corridor_extent(band)
    y_near = float(min(h - 2, r_bot))
    while y_near > r_top and left[int(y_near)] < 0:
        y_near -= 1
    if left[int(y_near)] < 0:
        return None, None, None
    d_near = y_near - vp[1]
    if d_near <= 4:
        return None, None, None
    y_far = vp[1] + depth_ratio * d_near
    y_far = float(max(y_far, r_top + 0.01 * h, vp[1] + 0.06 * d_near))
    if y_far >= y_near - 8:
        y_far = y_near - max(8.0, 0.12 * h)

    xl, xr = _corridor_span_at(left, right, r_top, y_near, vp, d_near, w)
    pad = (xr - xl) * lateral_pad
    xl -= pad
    xr += pad
    if xr - xl < 8:
        return None, None, None
    xl_f = _project_row(vp, xl, y_near, y_far)
    xr_f = _project_row(vp, xr, y_near, y_far)

    src = np.float32([[xl, y_near], [xr, y_near], [xr_f, y_far], [xl_f, y_far]])
    dst = np.float32([[0, BEV_H - 1], [BEV_W - 1, BEV_H - 1], [BEV_W - 1, 0], [0, 0]])
    try:
        H = cv2.getPerspectiveTransform(src, dst)
        Hi = np.linalg.inv(H)
    except Exception:
        return None, None, None
    info = {"y_near": y_near, "y_far": y_far, "x_left": xl, "x_right": xr,
            "src": src.tolist()}
    return H, Hi, info

def warp_pts(H, pts):
    P = np.asarray(pts, dtype=np.float64).reshape(-1, 1, 2)
    if P.size == 0:
        return np.zeros((0, 2))
    out = cv2.perspectiveTransform(P.astype(np.float32), H.astype(np.float32))
    return out.reshape(-1, 2)

# ==============================================================================
#  SECTION 4 - LANE EXTRACTION IN BIRD'S-EYE SPACE
# ==============================================================================
def bev_column_profile(paint_bev):
    H, W = paint_bev.shape[:2]
    m = (paint_bev > 0).astype(np.float32)
    taper = np.linspace(1.0, 0.5, H).astype(np.float32)
    prof = (m * taper[:, None]).sum(axis=0)
    return _smooth1d(prof, sigma=max(1.5, W * 0.005))

def estimate_pitch(prof, W, peaks=None):
    lo, hi = W / 16.0, W / 1.15
    ac_p, ac_s = _dominant_period(prof, lo, hi)
    sp_p, sp_n = None, 0
    if peaks and len(peaks) >= 3:
        d = np.diff(np.asarray(peaks, dtype=np.float64))
        d = d[(d > lo) & (d < hi)]
        if d.size >= 2:
            med = float(np.median(d))
            d = d[(d > 0.55 * med) & (d < 1.75 * med)]
            if d.size >= 2:
                sp_p, sp_n = float(np.median(d)), int(d.size)
    if ac_p and sp_p:
        if abs(ac_p - sp_p) / max(ac_p, sp_p) < 0.28:
            return 0.45 * ac_p + 0.55 * sp_p, max(ac_s, 0.5)
        return (sp_p, 0.5) if sp_n >= 3 else (ac_p, ac_s)
    if sp_p:
        return sp_p, 0.45
    if ac_p:
        return ac_p, ac_s
    return None, 0.0

def fit_lattice(prof, W, pitch0, n_hint=None):
    if pitch0 is None or pitch0 <= 2:
        return [], 0.0, None
    idx = np.arange(W)
    if n_hint and n_hint >= 1:
        pitch_lo = pitch_hi = W / float(n_hint)
        pitches = np.array([W / float(n_hint)])
    else:
        pitches = np.linspace(pitch0 * 0.80, pitch0 * 1.22, 36)
    best = (-1.0, None, None)
    for p in pitches:
        if p < 6:
            continue
        for ph in np.arange(0.0, p, max(0.8, p / 36.0)):
            us = np.arange(ph, W - 0.5, p)
            if us.size < 2:
                continue
            vals = np.interp(us, idx, prof)
            support = float(vals.mean()) * (1.0 + 0.015 * us.size)
            if support > best[0]:
                best = (support, float(p), us.copy())
    if best[1] is None:
        return [], 0.0, None
    _, pitch, us = best

    peaks = _find_peaks(prof, min_dist=max(3, int(pitch * 0.45)),
                        min_height=float(prof.mean() + 0.35 * prof.std()))
    snapped, snapped_n = [], 0
    tol = 0.22 * pitch
    for u in us:
        if peaks:
            near = min(peaks, key=lambda q: abs(q - u))
            if abs(near - u) <= tol:
                snapped.append(float(near))
                snapped_n += 1
                continue
        snapped.append(float(u))
    conf = _clamp(snapped_n / max(2.0, len(snapped)), 0.0, 1.0)
    return snapped, conf, pitch

def road_support_profile(band_bev):
    W = band_bev.shape[1]
    m = (band_bev > 0).astype(np.float32).mean(axis=0)
    return _smooth1d(m, sigma=max(1.5, W * 0.004))

def trim_to_road(bnds, road_prof, min_cover=0.45):
    if len(bnds) < 3 or road_prof is None:
        return list(bnds), 0
    W = road_prof.shape[0]
    idx = np.arange(W)
    b = list(bnds)

    def lane_cover(u0, u1):
        lo, hi = sorted((u0, u1))
        lo = _clamp(lo + 0.18 * (hi - lo), 0, W - 1)
        hi = _clamp(hi - 0.18 * (hi - lo), 0, W - 1)
        if hi <= lo:
            return float(np.interp(0.5 * (u0 + u1), idx, road_prof))
        us = np.linspace(lo, hi, 12)
        return float(np.mean(np.interp(us, idx, road_prof)))

    dropped = 0
    while len(b) >= 3 and lane_cover(b[0], b[1]) < min_cover:
        b.pop(0)
        dropped += 1
    while len(b) >= 3 and lane_cover(b[-2], b[-1]) < min_cover:
        b.pop()
        dropped += 1
    return b, dropped

def fit_boundary_curves(paint_bev, u_seeds, n_win=14, win_frac=0.055):
    H, W = paint_bev.shape[:2]
    m = (paint_bev > 0).astype(np.float32)
    win_h = max(4, H // n_win)
    win_w = max(4, int(W * win_frac))
    per_line = []
    for u0 in u_seeds:
        u = float(u0)
        us, vs, wts = [], [], []
        for k in range(n_win - 1, -1, -1):
            v0, v1 = k * win_h, min(H, (k + 1) * win_h)
            lo, hi = int(max(0, u - win_w)), int(min(W, u + win_w + 1))
            if hi - lo < 2:
                break
            sub = m[v0:v1, lo:hi]
            s = float(sub.sum())
            if s > max(2.0, 0.012 * (v1 - v0) * (hi - lo)):
                cols = sub.sum(axis=0)
                cu = lo + float((cols * np.arange(cols.size)).sum() / max(cols.sum(), 1e-6))
                us.append(cu)
                vs.append(0.5 * (v0 + v1))
                wts.append(s)
                u = 0.45 * u + 0.55 * cu
        per_line.append((np.array(vs), np.array(us), np.array(wts), float(u0)))

    A_list, B_list, w_list = [], [], []
    for vs, us, wts, _ in per_line:
        if vs.size >= 5 and (vs.max() - vs.min()) > 0.4 * H:
            try:
                c = np.polyfit(vs, us, 2, w=np.sqrt(wts))
                if abs(c[0]) < 4.0 / max(1.0, H):
                    A_list.append(c[0])
                    B_list.append(c[1])
                    w_list.append(float(wts.sum()))
            except Exception:
                pass
    if A_list:
        wv = np.array(w_list)
        A = float(np.sum(np.array(A_list) * wv) / wv.sum())
        B = float(np.sum(np.array(B_list) * wv) / wv.sum())
    else:
        A, B = 0.0, 0.0

    curves = []
    vv = np.arange(H, dtype=np.float64)
    shape = A * vv ** 2 + B * vv
    for vs, us, wts, u0 in per_line:
        if vs.size >= 2:
            C = float(np.average(us - (A * vs ** 2 + B * vs), weights=np.maximum(wts, 1e-6)))
            sup = float(vs.size) / n_win
        else:
            C = float(u0 - (A * (H - 1) ** 2 + B * (H - 1)))
            sup = 0.0
        curves.append({"A": A, "B": B, "C": C, "support": sup})

    curves.sort(key=lambda c: c["C"])
    for i in range(1, len(curves)):
        if curves[i]["C"] <= curves[i - 1]["C"] + 2.0:
            curves[i]["C"] = curves[i - 1]["C"] + 2.0
    for c in curves:
        c["u_bottom"] = float(c["A"] * (H - 1) ** 2 + c["B"] * (H - 1) + c["C"])
    return curves, (A, B)

def curve_points_bev(curve, n=48, h=BEV_H):
    v = np.linspace(0, h - 1, n)
    u = curve["A"] * v ** 2 + curve["B"] * v + curve["C"]
    return np.stack([u, v], axis=1)

def trajectory_lane_centres(tracks_bev, W, v_lo=0.10, v_hi=1.0):
    prof = np.zeros(W, np.float64)
    ntr = 0
    for pts in tracks_bev.values():
        P = np.asarray(pts, dtype=np.float64)
        if P.shape[0] < 3:
            continue
        sel = (P[:, 1] >= v_lo * BEV_H) & (P[:, 1] <= v_hi * BEV_H) & \
              (P[:, 0] >= 0) & (P[:, 0] < W)
        P = P[sel]
        if P.shape[0] < 3:
            continue
        wgt = 1.0 / P.shape[0]
        u = np.clip(P[:, 0].astype(int), 0, W - 1)
        np.add.at(prof, u, wgt)
        ntr += 1
    if ntr < 2:
        return [], None, 0.0, prof, ntr
    prof = _smooth1d(prof, sigma=max(2.5, W * 0.012))
    peaks = _find_peaks(prof, min_dist=max(5, int(W / 16)),
                        min_height=float(prof.mean() + 0.30 * prof.std()))
    pitch, strength = estimate_pitch(prof, W, peaks)
    conf = _clamp(ntr / 25.0, 0.0, 1.0) * (0.5 + 0.5 * float(strength))
    return [float(p) for p in peaks], pitch, conf, prof, ntr

def fuse_lane_geometry(paint_bnds, paint_pitch, paint_conf,
                       centres, traj_pitch, traj_conf, W, n_hint=None):
    notes = []
    bnds = [float(b) for b in (paint_bnds or [])]
    pitch = paint_pitch

    if bnds and centres and len(centres) >= 2:
        mids = [0.5 * (bnds[i] + bnds[i + 1]) for i in range(len(bnds) - 1)]
        if mids:
            err_mid = float(np.mean([min(abs(c - m) for m in mids) for c in centres]))
            err_bnd = float(np.mean([min(abs(c - b) for b in bnds) for c in centres]))
            p = pitch or (mids[1] - mids[0] if len(mids) > 1 else W)
            if err_bnd < err_mid * 0.62 and p > 8:
                bnds = [b + 0.5 * p for b in bnds]
                bnds = [b for b in bnds if -0.05 * W <= b <= 1.05 * W]
                if bnds and bnds[0] > 0.5 * p:
                    bnds.insert(0, bnds[0] - p)
                notes.append("lattice was half a lane out of phase - corrected "
                             "against vehicle paths")
            else:
                notes.append("marking lattice agrees with vehicle paths")

    if not bnds and centres and len(centres) >= 1:
        cs = sorted(centres)
        p = traj_pitch or (np.median(np.diff(cs)) if len(cs) > 1 else W * 0.25)
        p = float(max(8.0, p))
        bnds = [cs[0] - p / 2.0]
        for i in range(len(cs) - 1):
            bnds.append(0.5 * (cs[i] + cs[i + 1]))
        bnds.append(cs[-1] + p / 2.0)
        pitch = p
        notes.append(f"no usable markings - lanes inferred from {len(cs)} "
                     f"traffic streams")

    if bnds and centres and pitch and pitch > 8:
        lo, hi = min(bnds), max(bnds)
        guard = 0.35 * pitch
        while centres and min(centres) < lo - guard and lo - pitch > -0.15 * W:
            lo -= pitch
            bnds.insert(0, lo)
            notes.append("extended lanes to the left to cover observed traffic")
        while centres and max(centres) > hi + guard and hi + pitch < 1.15 * W:
            hi += pitch
            bnds.append(hi)
            notes.append("extended lanes to the right to cover observed traffic")

    if n_hint and len(bnds) != n_hint + 1 and pitch:
        lo = min(bnds) if bnds else 0.0
        span = W - lo
        p = span / float(n_hint)
        bnds = [lo + i * p for i in range(n_hint + 1)]
        pitch = p
        notes.append(f"lane count forced to {n_hint} by operator override")

    bnds = sorted(b for b in bnds if -0.10 * W <= b <= 1.10 * W)
    dedup = []
    for b in bnds:
        if not dedup or b - dedup[-1] > max(6.0, 0.25 * (pitch or 40)):
            dedup.append(b)
    conf = max(paint_conf * (1.0 if paint_bnds else 0.0), traj_conf * 0.9)
    return dedup, pitch, _clamp(conf, 0.0, 1.0), notes

# ==============================================================================
#  SECTION 5 - MONOCULAR METRIC CALIBRATION
# ==============================================================================
class PerspectiveCalib:
    def __init__(self, vp, y_h, a_slope, lane_w_m, k=None, f=None,
                 scale_src="unknown", conf=0.0, frame_w=1280, frame_h=720,
                 width_fit_r2=0.0):
        self.vx, self.vy = float(vp[0]), float(vp[1])
        self.y_h = float(y_h)
        self.a = float(a_slope)
        self.lane_w_m = float(lane_w_m)
        self.f = float(f) if f else FOV_FALLBACK_FACTOR * frame_w
        self.k = float(k) if k else self.f * self.lane_w_m / max(self.a, 1e-6)
        self.scale_src = scale_src
        self.conf = float(conf)
        self.frame_w, self.frame_h = int(frame_w), int(frame_h)
        self.width_fit_r2 = float(width_fit_r2)
        self._build_rows()

    def _build_rows(self):
        h = self.frame_h
        y = np.arange(h, dtype=np.float64)
        d = y - self.y_h
        self.valid = d > max(3.0, 0.035 * h)
        d_safe = np.where(self.valid, d, np.nan)
        self.row_Z = self.k / d_safe
        self.row_wpx = self.a * d_safe
        self.row_lat = self.lane_w_m / np.maximum(self.row_wpx, 1e-6)

    def to_metric(self, x, y):
        yi = int(_clamp(int(round(y)), 0, self.frame_h - 1))
        if not self.valid[yi]:
            return np.nan, np.nan
        Z = float(self.row_Z[yi])
        X = (float(x) - self.vx) * Z / self.f
        return X, Z

    def lat_m_per_px(self, y):
        yi = int(_clamp(int(round(y)), 0, self.frame_h - 1))
        v = self.row_lat[yi]
        return float(v) if np.isfinite(v) else float("nan")

    def lane_width_px(self, y):
        yi = int(_clamp(int(round(y)), 0, self.frame_h - 1))
        v = self.row_wpx[yi]
        return float(v) if np.isfinite(v) else float("nan")

    def is_valid_row(self, y):
        yi = int(_clamp(int(round(y)), 0, self.frame_h - 1))
        return bool(self.valid[yi])

    def rescaled(self, s):
        if abs(s - 1.0) < 1e-6:
            return self
        return PerspectiveCalib(
            (self.vx * s, self.vy * s), self.y_h * s, self.a, self.lane_w_m,
            k=self.k * s, f=self.f * s, scale_src=self.scale_src, conf=self.conf,
            frame_w=int(round(self.frame_w * s)), frame_h=int(round(self.frame_h * s)),
            width_fit_r2=self.width_fit_r2)

    def to_dict(self):
        return {"vx": self.vx, "vy": self.vy, "y_h": self.y_h, "a": self.a,
                "lane_w_m": self.lane_w_m, "k": self.k, "f": self.f,
                "scale_src": self.scale_src, "conf": self.conf,
                "frame_w": self.frame_w, "frame_h": self.frame_h,
                "width_fit_r2": self.width_fit_r2}

    @staticmethod
    def from_dict(d):
        return PerspectiveCalib((d["vx"], d["vy"]), d["y_h"], d["a"], d["lane_w_m"],
                                k=d.get("k"), f=d.get("f"),
                                scale_src=d.get("scale_src", "unknown"),
                                conf=d.get("conf", 0.0),
                                frame_w=d.get("frame_w", 1280),
                                frame_h=d.get("frame_h", 720),
                                width_fit_r2=d.get("width_fit_r2", 0.0))

    def describe(self):
        hfov = 2.0 * math.degrees(math.atan(0.5 * self.frame_w / max(self.f, 1e-6)))
        return {
            "Vanishing point": f"({self.vx:.0f}, {self.vy:.0f}) px",
            "Horizon row": f"{self.y_h:.0f} px",
            "Lane width model": f"{self.a:.4f} px per row  (R^2 {self.width_fit_r2:.3f})",
            "Assumed lane width": f"{self.lane_w_m:.2f} m",
            "Focal length": f"{self.f:.0f} px  (~{hfov:.0f} deg HFOV)",
            "Depth constant k": f"{self.k:.0f} px*m",
            "Longitudinal scale": self.scale_src,
        }

def fit_width_model(boundary_rows, vy, frame_h, pairs=None):
    ys, ws = [], []
    if pairs:
        spans = list(pairs)
    else:
        nb = len(boundary_rows)
        if nb < 2:
            return None, 0.0, None
        spans = [(boundary_rows[i], boundary_rows[i + 1]) for i in range(nb - 1)]
    if not spans:
        return None, 0.0, None
    for y in range(int(frame_h * 0.25), frame_h, max(1, frame_h // 120)):
        gaps = []
        for lrow, rrow in spans:
            x0, x1 = lrow[y], rrow[y]
            if np.isfinite(x0) and np.isfinite(x1) and x1 > x0:
                gaps.append(x1 - x0)
        if gaps:
            ys.append(float(y))
            ws.append(float(np.median(gaps)))
    if len(ys) < 8:
        return None, 0.0, None
    ys = np.asarray(ys)
    ws = np.asarray(ws)
    d = ys - vy
    ok = d > max(3.0, 0.03 * frame_h)
    if ok.sum() < 6:
        return None, 0.0, None
    a = float(np.sum(ws[ok] * d[ok]) / np.sum(d[ok] ** 2))
    try:
        m, c = np.polyfit(ys, ws, 1)
        pred = m * ys + c
        ss_res = float(np.sum((ws - pred) ** 2))
        ss_tot = float(np.sum((ws - ws.mean()) ** 2))
        r2 = 1.0 - ss_res / max(ss_tot, 1e-9)
        y_h_free = float(-c / m) if abs(m) > 1e-9 else None
    except Exception:
        r2, y_h_free = 0.0, None
    return a, _clamp(r2, 0.0, 1.0), y_h_free

def dash_period_scale(paint, boundary_rows, y_h, y_near, y_far, std_period_m,
                      frame_shape):
    if not std_period_m or std_period_m <= 0:
        return None, 0.0, 0
    h, w = frame_shape[:2]
    y_lo = max(y_h + 0.42 * (y_near - y_h), y_far)
    y_hi = y_near
    if y_hi - y_lo < 0.10 * h:
        return None, 0.0, 0
    rows = np.arange(int(y_lo), int(y_hi) + 1)
    if rows.size < 40:
        return None, 0.0, 0
    q = 1.0 / (rows - y_h)
    q_lo, q_hi = float(q.min()), float(q.max())
    ngrid = 512
    qg = np.linspace(q_lo, q_hi, ngrid)
    dq = (q_hi - q_lo) / (ngrid - 1)

    ks, strengths = [], []
    for brow in boundary_rows:
        xs = brow[rows]
        good = np.isfinite(xs)
        if good.sum() < 0.7 * rows.size:
            continue
        sig = np.zeros(rows.size, np.float32)
        tol = 2
        for i, (yy, xx) in enumerate(zip(rows, xs)):
            if not np.isfinite(xx):
                continue
            x0 = int(_clamp(int(round(xx)) - tol, 0, w - 1))
            x1 = int(_clamp(int(round(xx)) + tol + 1, 1, w))
            sig[i] = 1.0 if paint[int(yy), x0:x1].any() else 0.0
        fill = float(sig.mean())
        if fill < 0.08 or fill > 0.92:
            continue
        order = np.argsort(q)
        sg = np.interp(qg, q[order], sig[order])
        lag, strength = _dominant_period(sg, ngrid / 22.0, ngrid / 2.6)
        if lag is None or strength < 0.28:
            continue
        dq_period = lag * dq
        if dq_period <= 1e-12:
            continue
        ks.append(std_period_m / dq_period)
        strengths.append(strength)
    if not ks:
        return None, 0.0, 0
    ks = np.asarray(ks)
    med = float(np.median(ks))
    keep = np.abs(ks - med) < 0.28 * max(med, 1e-6)
    if keep.sum() == 0:
        keep = np.ones_like(ks, bool)
    return float(np.median(ks[keep])), float(np.mean(np.asarray(strengths)[keep])), int(keep.sum())

# ==============================================================================
#  SECTION 6 - RUNTIME LANE MODEL
# ==============================================================================
def _poly_edges(lane):
    if lane.get("left") and lane.get("right"):
        return (np.asarray(lane["left"], np.float64),
                np.asarray(lane["right"], np.float64))
    P = np.asarray(lane["polygon"], np.float64)
    if P.shape[0] == 4:
        cy = P[:, 1].mean()
        cx = P[:, 0].mean()
        bot = P[P[:, 1] >= cy]
        top = P[P[:, 1] < cy]
        if bot.shape[0] != 2 or top.shape[0] != 2:
            order = np.argsort(P[:, 1])
            top, bot = P[order[:2]], P[order[2:]]
        bl = bot[np.argmin(bot[:, 0])]
        br = bot[np.argmax(bot[:, 0])]
        tl = top[np.argmin(top[:, 0])]
        tr = top[np.argmax(top[:, 0])]
        _ = cx
        return np.array([bl, tl]), np.array([br, tr])
    half = P.shape[0] // 2
    return P[:half], P[half:][::-1]

def _rows_from_polyline(pl, frame_h, vp=None):
    P = np.asarray(pl, np.float64)
    if P.shape[0] < 2:
        return np.full(frame_h, np.nan)
    order = np.argsort(P[:, 1])
    ys, xs = P[order, 1], P[order, 0]
    rows = np.arange(frame_h, dtype=np.float64)
    out = np.interp(rows, ys, xs)
    lo, hi = ys[0], ys[-1]
    if vp is not None:
        for r in range(frame_h):
            if rows[r] < lo:
                out[r] = _project_row(vp, xs[0], lo, rows[r])
            elif rows[r] > hi:
                out[r] = _project_row(vp, xs[-1], hi, rows[r])
    return out

def _full_span_polyline(row_x, vp, frame_h, n=60):
    y_top = 0
    ys = np.unique(np.linspace(y_top, frame_h - 1, n).astype(int))
    pts = [[float(row_x[y]), float(y)] for y in ys if np.isfinite(row_x[y])]
    if len(pts) < 2:
        return np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    return np.asarray(pts, dtype=np.float64)

class LaneModel:
    def __init__(self, lanes, calib, frame_shape):
        self.h, self.w = int(frame_shape[0]), int(frame_shape[1])
        self.calib = calib
        self.names = list(lanes.keys())
        self.lanes = lanes
        vp = (calib.vx, calib.vy) if calib else None
        self.left_rows, self.right_rows = [], []
        for nm in self.names:
            L, R = _poly_edges(lanes[nm])
            self.left_rows.append(_rows_from_polyline(L, self.h, vp))
            self.right_rows.append(_rows_from_polyline(R, self.h, vp))
        self.flow = [int(lanes[nm].get("flow", 0)) for nm in self.names]
        self.flow_trusted = [bool(lanes[nm].get("flow_trusted", False)) for nm in self.names]
        self.support = [float(lanes[nm].get("support", 0.0)) for nm in self.names]
        self.kind = [str(lanes[nm].get("kind", "travel")) for nm in self.names]
        self.index_map = np.full((self.h, self.w), -1, np.int16)
        for i, nm in enumerate(self.names):
            poly = np.asarray(lanes[nm]["polygon"], np.int32)
            if poly.shape[0] >= 3:
                cv2.fillPoly(self.index_map, [poly], int(i))
        self.boundary_rows = self._boundary_rows()
        self.boundary_types = self._boundary_types()

    def _boundary_types(self):
        if not self.names:
            return []
        t = [str(self.lanes[self.names[0]].get("left_type", "unknown"))]
        for nm in self.names:
            t.append(str(self.lanes[nm].get("right_type", "unknown")))
        return t

    def _boundary_rows(self):
        rows = []
        if not self.names:
            return rows
        rows.append(self.left_rows[0])
        for i in range(len(self.names)):
            rows.append(self.right_rows[i])
        return rows

    def lane_at(self, x, y):
        xi = int(_clamp(int(round(x)), 0, self.w - 1))
        yi = int(_clamp(int(round(y)), 0, self.h - 1))
        return int(self.index_map[yi, xi])

    def name(self, i):
        return self.names[i] if 0 <= i < len(self.names) else "off-lane"

    def n_travel(self):
        return sum(1 for k in self.kind if k == "travel")

    def edges_at(self, i, y):
        yi = int(_clamp(int(round(y)), 0, self.h - 1))
        return float(self.left_rows[i][yi]), float(self.right_rows[i][yi])

    def lateral_offset_m(self, i, x, y):
        xl, xr = self.edges_at(i, y)
        if not (np.isfinite(xl) and np.isfinite(xr)) or xr - xl < 1e-3:
            return float("nan"), float("nan")
        c = 0.5 * (xl + xr)
        wpx = xr - xl
        m_per_px = self.calib.lane_w_m / wpx if self.calib else 1.0
        return (float(x) - c) * m_per_px, wpx * m_per_px

    def nearest_boundary_gap_m(self, x, y):
        yi = int(_clamp(int(round(y)), 0, self.h - 1))
        best = None
        m_per_px = self.calib.lat_m_per_px(y) if self.calib else 1.0
        for br in self.boundary_rows:
            bx = br[yi]
            if not np.isfinite(bx):
                continue
            d = abs(float(x) - float(bx))
            if best is None or d < best:
                best = d
        if best is None or not np.isfinite(m_per_px):
            return float("nan")
        return best * m_per_px

    def draw(self, img, alpha=0.16, thick=2):
        out = img
        ov = img.copy()
        for i, nm in enumerate(self.names):
            poly = np.asarray(self.lanes[nm]["polygon"], np.int32)
            if poly.shape[0] < 3:
                continue
            shoulder = self.kind[i] != "travel"
            fill = (90, 90, 90) if shoulder else (255, 200, 0)
            edge = (120, 120, 120) if shoulder else (255, 150, 0)
            cv2.fillPoly(ov, [poly], fill)
            cv2.polylines(out, [poly], True, edge, thick, cv2.LINE_AA)
        cv2.addWeighted(ov, alpha, out, 1 - alpha, 0, out)
        for i, nm in enumerate(self.names):
            yi = int(self.h * 0.80)
            xl, xr = self.edges_at(i, yi)
            if not (np.isfinite(xl) and np.isfinite(xr)):
                continue
            cx = int(0.5 * (xl + xr))
            tag = nm if self.kind[i] == "travel" else f"{nm} (shoulder)"
            if self.flow[i]:
                tag += " v" if self.flow[i] > 0 else " ^"
            _label(out, tag, cx - 30, yi, (255, 150, 0), 0.45)
        return out

# ==============================================================================
#  SECTION 7 - CALIBRATION ORCHESTRATOR
# ==============================================================================
def prescan_tracks(video, conf=0.25, imgsz=640, max_frames=PRESCAN_MAX_FRAMES,
                   target_fps=PRESCAN_TARGET_FPS, progress=None):
    meta = video_meta(video)
    stride = max(1, int(round(meta["fps"] / max(1.0, target_fps))))
    model, _minfo = get_model()
    if model is None:
        return {"tracks": {}, "classes": {}, "box_w": {}, "stride": stride,
                "fps": meta["fps"], "frames_used": 0}
    tracks = defaultdict(list)
    cls_votes = defaultdict(Counter)
    box_w = defaultdict(list)
    gen = None
    try:
        gen = model.track(source=video, stream=True, conf=conf, iou=0.5,
                          tracker=ensure_tracker_cfg(), device=get_device(),
                          half=(get_device() == 0), vid_stride=stride,
                          imgsz=imgsz, verbose=False)
        for fi, r in enumerate(gen):
            if fi >= max_frames:
                break
            if progress is not None and fi % 10 == 0:
                try:
                    progress(min(0.45, 0.05 + 0.40 * fi / max(1, max_frames)),
                             desc=f"Learning the road from traffic ({fi}/{max_frames})")
                except Exception:
                    pass
            if r.boxes is None or r.boxes.id is None:
                continue
            xy = r.boxes.xyxy.cpu().numpy()
            cl = r.boxes.cls.cpu().numpy()
            ids = r.boxes.id.cpu().numpy()
            for b, c, t in zip(xy, cl, ids):
                name = model.names[int(c)]
                if not _is_vehicle(name):
                    continue
                x1, y1, x2, y2 = [float(v) for v in b[:4]]
                tracks[int(t)].append(((x1 + x2) * 0.5, y2, fi))
                cls_votes[int(t)][name] += 1
                box_w[int(t)].append((y2, x2 - x1))
    except Exception:
        pass
    finally:
        try:
            if gen is not None and hasattr(gen, "close"):
                gen.close()
        except Exception:
            pass
    return {"tracks": dict(tracks), "classes": dict(cls_votes), "box_w": dict(box_w),
            "stride": stride, "fps": meta["fps"], "frames_used": min(max_frames, 10 ** 9)}

def _scale_tracks(tracks, s):
    if abs(s - 1.0) < 1e-6:
        return tracks
    return {t: [(x * s, y * s, f) for (x, y, f) in pts] for t, pts in tracks.items()}

def _speed_p85_kmh(tracks, calib, fps, stride):
    sp = []
    dt = stride / max(1e-6, fps)
    for pts in tracks.values():
        if len(pts) < 4:
            continue
        mets = []
        for (x, y, f) in pts:
            X, Z = calib.to_metric(x, y)
            if np.isfinite(X) and np.isfinite(Z):
                mets.append((X, Z, f))
        if len(mets) < 4:
            continue
        (X0, Z0, f0), (X1, Z1, f1) = mets[0], mets[-1]
        nf = max(1, f1 - f0)
        d = math.hypot(X1 - X0, Z1 - Z0)
        v = d / (nf * dt) * 3.6
        if 0.5 < v < 400:
            sp.append(v)
    if len(sp) < 3:
        return None
    return float(np.percentile(sp, 85))

def calibrate_video(video, engine=ENGINE_MODES[0], lane_width_m=DEFAULT_LANE_WIDTH_M,
                    marking_std="Auto (measure & guess)", n_lane_hint=0,
                    det_conf=0.25, progress=None):
    t0 = time.time()
    rep = {"steps": [], "warnings": [], "notes": []}

    def step(msg):
        rep["steps"].append(msg)

    def _pg(frac, desc):
        if progress is not None:
            try:
                progress(frac, desc=desc)
            except Exception:
                pass

    _pg(0.02, "Sampling frames")
    frames, scale, meta = sample_frames(video)
    if len(frames) < 4:
        return None, "Could not read enough frames from this video."
    ah, aw = frames[0].shape[:2]
    fw = int(round(aw / scale)) if scale > 0 else aw
    fh = int(round(ah / scale)) if scale > 0 else ah
    step(f"Sampled {len(frames)} frames across {meta['dur']:.1f}s "
         f"({meta['w']}x{meta['h']} @ {meta['fps']:.1f} fps); "
         f"analysing at {aw}x{ah}")

    _pg(0.06, "Building clean background")
    bg, act = temporal_stats(frames)
    act_m = activity_mask(act, bg.shape)
    band = road_corridor(bg, act_m)
    left, right, r_top, r_bot = corridor_extent(band)
    step(f"Road corridor spans rows {r_top}-{r_bot}; "
         f"width at bottom {int(right[r_bot] - left[r_bot])} px")
    moving = float(np.count_nonzero(act_m)) / float(ah * aw) if act_m is not None else 0.0
    if moving < 0.004:
        rep["warnings"].append("Very little motion in this clip - the road extent "
                               "came mostly from surface appearance, not traffic.")

    _pg(0.12, "Extracting lane markings")
    paint = paint_mask(bg, band)
    nvote = min(14, len(frames))
    votes = np.zeros((ah, aw), np.uint16)
    for fr in frames[:: max(1, len(frames) // nvote)][:nvote]:
        votes += (paint_mask(fr, band) > 0).astype(np.uint16)
    vote_mask = (votes >= max(2, int(0.35 * nvote))).astype(np.uint8) * 255
    paint_all = cv2.bitwise_or(paint, cv2.bitwise_and(vote_mask, band))
    paint_px = int(np.count_nonzero(paint_all))
    step(f"Marking pixels found: {paint_px} "
         f"({100.0 * paint_px / max(1, np.count_nonzero(band)):.2f}% of road area)")
    paint_ok = paint_px > max(120, 0.0004 * ah * aw)
    if not paint_ok:
        rep["warnings"].append("Barely any painted markings detected - falling back "
                               "to trajectory-derived lanes.")

    segs = segments_from_mask(paint_all, bg.shape)
    vp_p, c_p, n_p = ransac_vanishing_point(segs, bg.shape, r_top)
    if vp_p:
        step(f"Marking vanishing point ({vp_p[0]:.0f}, {vp_p[1]:.0f}) "
             f"from {n_p}/{len(segs)} segments, confidence {c_p:.2f}")
    else:
        step("No vanishing point from markings.")

    pres = None
    tracks_a = {}
    want_prescan = engine != ENGINE_MODES[1]
    if want_prescan:
        _pg(0.18, "Learning the road from traffic")
        try:
            pres = prescan_tracks(video, conf=det_conf, progress=progress)
            tracks_a = _scale_tracks(pres["tracks"], scale)
            step(f"Prescan tracked {len(tracks_a)} vehicles "
                 f"(every {pres['stride']} frames)")
        except Exception as e:
            rep["warnings"].append(f"Prescan unavailable ({e}); using markings only.")
            pres, tracks_a = None, {}
    vp_t, c_t, n_t = (None, 0.0, 0)
    if tracks_a:
        vp_t, c_t, n_t = vp_from_tracks({k: [(p[0], p[1]) for p in v]
                                         for k, v in tracks_a.items()}, bg.shape)
        if vp_t:
            step(f"Trajectory vanishing point ({vp_t[0]:.0f}, {vp_t[1]:.0f}) "
                 f"from {n_t} paths, confidence {c_t:.2f}")

    if engine == ENGINE_MODES[2]:
        vp_p, c_p = None, 0.0
    if engine == ENGINE_MODES[1]:
        vp_t, c_t = None, 0.0
    vp, vp_conf, vp_src = fuse_vps(vp_p, c_p, vp_t, c_t, bg.shape)
    if vp is None:
        vp = (aw * 0.5, max(0.0, r_top - 0.10 * ah))
        vp_conf, vp_src = 0.15, "fallback (frame geometry)"
        rep["warnings"].append("Vanishing point could not be measured; assumed one "
                               "above the road. Check the calibration overlay.")
    step(f"Vanishing point source: {vp_src}")

    _pg(0.55, "Warping to bird's-eye view")
    H, Hi, ipm = build_ipm(vp, band, bg.shape)
    if H is None:
        return None, ("Could not build a bird's-eye view - the road region or "
                      "vanishing point is degenerate. Try Manual Draw.")

    paint_bev = cv2.warpPerspective(paint_all, H, (BEV_W, BEV_H), flags=cv2.INTER_LINEAR)
    paint_bev = (paint_bev > 30).astype(np.uint8) * 255

    prof = bev_column_profile(paint_bev)
    pk = _find_peaks(prof, min_dist=max(4, BEV_W // 26),
                     min_height=float(prof.mean() + 0.35 * prof.std()))
    pitch_p, pstr = estimate_pitch(prof, BEV_W, pk)
    bnds_p, conf_p, pitch_p2 = ([], 0.0, None)
    if paint_ok and engine != ENGINE_MODES[2] and pitch_p:
        bnds_p, conf_p, pitch_p2 = fit_lattice(prof, BEV_W, pitch_p,
                                               n_hint=(n_lane_hint or None))
        step(f"Marking lattice: {len(bnds_p)} boundaries, pitch "
             f"{pitch_p2:.1f} px (periodicity {pstr:.2f}, snap {conf_p:.2f})")

    centres, pitch_t, conf_t, tprof, ntr = ([], None, 0.0, None, 0)
    tracks_bev = {}
    if tracks_a and engine != ENGINE_MODES[1]:
        for tid, pts in tracks_a.items():
            arr = warp_pts(H, [(p[0], p[1]) for p in pts])
            tracks_bev[tid] = arr
        centres, pitch_t, conf_t, tprof, ntr = trajectory_lane_centres(tracks_bev, BEV_W)
        if centres:
            step(f"Traffic streams: {len(centres)} lane centres from {ntr} vehicles"
                 + (f", pitch {pitch_t:.1f} px" if pitch_t else ""))

    bnds, pitch, geo_conf, fnotes = fuse_lane_geometry(
        bnds_p, pitch_p2, conf_p, centres, pitch_t, conf_t, BEV_W,
        n_hint=(n_lane_hint or None))
    rep["notes"].extend(fnotes)

    band_bev = cv2.warpPerspective(band, H, (BEV_W, BEV_H), flags=cv2.INTER_NEAREST)
    road_prof = road_support_profile(band_bev)
    if not n_lane_hint:
        bnds, n_trim = trim_to_road(bnds, road_prof)
        if n_trim:
            step(f"Trimmed {n_trim} lattice line(s) that fell outside the "
                 f"carriageway.")
    if len(bnds) < 2:
        n = n_lane_hint or 4
        bnds = [BEV_W * i / float(n) for i in range(n + 1)]
        pitch = BEV_W / float(n)
        rep["warnings"].append(f"Lane structure was not measurable; fell back to "
                               f"{n} equal lanes across the road.")
    step(f"Final lane structure: {len(bnds) - 1} lanes, {len(bnds)} boundaries")

    _pg(0.68, "Fitting lane boundaries")
    curves, (cA, cB) = fit_boundary_curves(paint_bev, bnds)
    if abs(cA) > 1e-9:
        step(f"Shared curvature model A={cA:.2e}, B={cB:.3f} "
             f"(road bends {'right' if cA > 0 else 'left'})")

    bnd_img = []
    for c in curves:
        p_bev = curve_points_bev(c, n=52)
        p_img = warp_pts(Hi, p_bev)
        bnd_img.append(p_img)
    bnd_rows = [_rows_from_polyline(p, ah, vp) for p in bnd_img]
    bnd_img = [_full_span_polyline(br, vp, ah) for br in bnd_rows]

    _pg(0.76, "Calibrating real-world scale")
    a_slope, r2, y_h_free = fit_width_model(bnd_rows, vp[1], ah)
    if a_slope is None or a_slope <= 1e-6:
        a_slope = max(1e-4, (right[r_bot] - left[r_bot]) /
                      max(1.0, (r_bot - vp[1])) / max(1, len(bnds) - 1))
        r2 = 0.0
        rep["warnings"].append("Lane-width model could not be fitted; lateral scale "
                               "is a rough estimate.")
    y_h = float(vp[1])
    if y_h_free is not None and abs(y_h_free - vp[1]) < 0.06 * ah:
        y_h = 0.5 * (y_h + y_h_free)
    step(f"Lane width grows {a_slope:.4f} px per row (parallelism R^2 {r2:.3f}); "
         f"horizon row {y_h:.0f}")
    if r2 < 0.80:
        rep["warnings"].append(f"Lane boundaries are only weakly parallel "
                               f"(R^2={r2:.2f}); the vanishing point may be off.")

    std_period = MARKING_STANDARDS.get(marking_std, (None, ""))[0]
    k_val, k_str, k_n = (None, 0.0, 0)
    if std_period is None:
        k_val, k_str, k_n = dash_period_scale(paint_all, bnd_rows, y_h, ipm["y_near"],
                                              ipm["y_far"], 9.0, bg.shape)
        if k_val:
            step(f"Dash rhythm detected on {k_n} boundary line(s) "
                 f"(clarity {k_str:.2f}); assuming a 9 m mark+gap period")
    elif std_period > 0:
        k_val, k_str, k_n = dash_period_scale(paint_all, bnd_rows, y_h, ipm["y_near"],
                                              ipm["y_far"], std_period, bg.shape)
        if k_val:
            step(f"Longitudinal scale locked to the {std_period:.1f} m marking "
                 f"period on {k_n} line(s) (clarity {k_str:.2f})")

    if k_val and k_str >= 0.28:
        f_try = a_slope * k_val / max(lane_width_m, 0.5)
        hfov = 2.0 * math.degrees(math.atan(0.5 * aw / max(f_try, 1e-6)))
        if not (F_PX_MIN_FACTOR * aw <= f_try <= F_PX_MAX_FACTOR * aw):
            step(f"Rejected the dash-derived scale: it implies a {hfov:.0f} deg "
                 f"field of view, outside anything a real camera produces.")
            rep["warnings"].append(
                f"The marking rhythm gave an impossible camera geometry "
                f"({hfov:.0f} deg field of view), which almost always means the "
                f"measured period is a multiple of the real one. Speeds now use an "
                f"assumed focal length and are approximate; set the correct marking "
                f"standard to fix them.")
            k_val, k_str = None, 0.0

    if k_val and k_str >= 0.28:
        f_px = a_slope * k_val / max(lane_width_m, 0.5)
        scale_src = f"dash period ({k_n} line{'s' if k_n != 1 else ''}, clarity {k_str:.2f})"
        conf_scale = _clamp(0.55 + 0.45 * k_str, 0, 1)
    else:
        f_px = FOV_FALLBACK_FACTOR * aw
        k_val = f_px * lane_width_m / max(a_slope, 1e-6)
        scale_src = f"assumed {FOV_FALLBACK_FACTOR:.2f}xW focal length (approximate)"
        conf_scale = 0.35
        rep["warnings"].append("No usable dash rhythm - speeds rely on an assumed "
                               "camera focal length and should be treated as "
                               "indicative. Pick your marking standard, or set a "
                               "known speed reference, to make them exact.")

    calib_a = PerspectiveCalib(vp, y_h, a_slope, lane_width_m, k=k_val, f=f_px,
                               scale_src=scale_src, conf=conf_scale,
                               frame_w=aw, frame_h=ah, width_fit_r2=r2)

    if tracks_a:
        p85 = _speed_p85_kmh(tracks_a, calib_a, pres["fps"], pres["stride"])
        if p85 is not None:
            step(f"Prescan 85th-percentile speed: {p85:.0f} km/h")
            if not (6.0 <= p85 <= 190.0):
                f_px = FOV_FALLBACK_FACTOR * aw
                k_val = f_px * lane_width_m / max(a_slope, 1e-6)
                calib_a = PerspectiveCalib(vp, y_h, a_slope, lane_width_m, k=k_val,
                                           f=f_px, scale_src="assumed focal length "
                                           "(dash estimate rejected as implausible)",
                                           conf=0.30, frame_w=aw, frame_h=ah,
                                           width_fit_r2=r2)
                rep["warnings"].append(f"Dash-derived scale implied {p85:.0f} km/h "
                                       f"typical traffic, which is implausible; "
                                       f"reverted to the assumed focal length.")

    btypes = classify_boundaries(paint_all, bnd_rows, y_h, ipm["y_near"],
                                 ipm["y_far"], bg.shape)
    tcount = Counter(b["type"] for b in btypes)
    step("Boundary lines: " + ", ".join(f"{v} {k}" for k, v in tcount.items()))

    lanes = {}
    for i in range(len(bnd_img) - 1):
        L = bnd_img[i]
        R = bnd_img[i + 1]
        poly = np.vstack([L, R[::-1]])
        lanes[f"Lane {i + 1}"] = {
            "polygon": [[int(round(p[0])), int(round(p[1]))] for p in poly],
            "left": [[float(p[0]), float(p[1])] for p in L],
            "right": [[float(p[0]), float(p[1])] for p in R],
            "left_type": btypes[i]["type"] if i < len(btypes) else "unknown",
            "right_type": btypes[i + 1]["type"] if i + 1 < len(btypes) else "unknown",
            "flow": 0, "support": 0.0, "kind": "travel",
        }
    if not lanes:
        return None, "Lane construction failed."

    lm_a = LaneModel(lanes, calib_a, (ah, aw))
    supp = Counter()
    flowv = defaultdict(Counter)
    for tid, pts in tracks_a.items():
        if len(pts) < 3:
            continue
        ids = [lm_a.lane_at(p[0], p[1]) for p in pts]
        ids = [i for i in ids if i >= 0]
        if not ids:
            continue
        main = Counter(ids).most_common(1)[0][0]
        supp[main] += 1
        dy = pts[-1][1] - pts[0][1]
        if abs(dy) > 0.03 * ah:
            flowv[main][1 if dy > 0 else -1] += 1
    tot_tracks = max(1, sum(supp.values()))
    for i, nm in enumerate(lm_a.names):
        lanes[nm]["support"] = float(supp.get(i, 0))
        if flowv[i]:
            lanes[nm]["flow"] = int(flowv[i].most_common(1)[0][0])
    if tot_tracks >= 12:
        n = len(lm_a.names)
        for i, nm in enumerate(lm_a.names):
            if supp.get(i, 0) == 0 and i in (0, n - 1):
                lanes[nm]["kind"] = "shoulder"
                rep["notes"].append(f"{nm} carried no traffic and sits at the road "
                                    f"edge - treated as a shoulder.")
    global_flow = Counter()
    for i in range(len(lm_a.names)):
        for k, v in flowv[i].items():
            global_flow[k] += v
    gf = global_flow.most_common(1)[0][0] if global_flow else 1
    for i, nm in enumerate(lm_a.names):
        if sum(flowv[i].values()) < WRONGWAY_MIN_SUPPORT:
            lanes[nm]["flow_trusted"] = False
            if lanes[nm]["flow"] == 0:
                lanes[nm]["flow"] = int(gf)
        else:
            lanes[nm]["flow_trusted"] = True

    inv_s = 1.0 / scale if scale > 0 else 1.0
    if abs(inv_s - 1.0) > 1e-6:
        for nm in lanes:
            lanes[nm]["polygon"] = [[int(round(x * inv_s)), int(round(y * inv_s))]
                                    for x, y in lanes[nm]["polygon"]]
            lanes[nm]["left"] = [[x * inv_s, y * inv_s] for x, y in lanes[nm]["left"]]
            lanes[nm]["right"] = [[x * inv_s, y * inv_s] for x, y in lanes[nm]["right"]]
    calib_full = calib_a.rescaled(inv_s)
    calib_full.frame_w, calib_full.frame_h = fw, fh
    calib_full._build_rows()

    # Create diagnostics but without top-left text to keep video clean natively
    diag = make_diagnostics(bg, band, paint_all, vp, lanes, calib_a, scale,
                            prof, tprof, bnds)
    cv2.imwrite(DIAG_IMAGE, diag)
    cv2.imwrite(BG_FRAME, bg)

    out = {
        "schema": CALIB_SCHEMA,
        "video_sig": _file_sig(video),
        "frame_w": fw, "frame_h": fh,
        "analysis_scale": scale,
        "lanes": lanes,
        "perspective": calib_full.to_dict(),
        "engine": engine,
        "marking_std": marking_std,
        "vp_source": vp_src,
        "vp_conf": vp_conf,
        "geometry_conf": geo_conf,
        "pitch_bev": pitch,
        "n_lanes": len(lanes),
        "boundary_types": [b["type"] for b in btypes],
        "boundary_fill": [round(b["fill"], 3) for b in btypes],
        "report": rep,
        "elapsed": time.time() - t0,
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    _pg(0.98, "Calibration complete")
    return out, (f"Calibrated in {out['elapsed']:.1f}s - {len(lanes)} lanes, "
                 f"vanishing point from {vp_src}.")

def classify_boundaries(paint, bnd_rows, y_h, y_near, y_far, frame_shape):
    h, w = frame_shape[:2]
    y_lo = max(y_h + 0.42 * (y_near - y_h), y_far)
    rows = np.arange(int(max(0, y_lo)), int(min(h - 1, y_near)) + 1)
    out = []
    if rows.size < 20:
        return [{"type": "unknown", "fill": 0.0, "rhythm": 0.0} for _ in bnd_rows]
    q = 1.0 / np.maximum(rows - y_h, 1e-6)
    for brow in bnd_rows:
        xs = brow[rows]
        sig = np.zeros(rows.size, np.float32)
        for i, (yy, xx) in enumerate(zip(rows, xs)):
            if not np.isfinite(xx):
                continue
            x0 = int(_clamp(int(round(xx)) - 2, 0, w - 1))
            x1 = int(_clamp(int(round(xx)) + 3, 1, w))
            sig[i] = 1.0 if paint[int(yy), x0:x1].any() else 0.0
        fill = float(sig.mean())
        rhythm = 0.0
        if 0.08 < fill < 0.92:
            order = np.argsort(q)
            qg = np.linspace(q.min(), q.max(), 384)
            sg = np.interp(qg, q[order], sig[order])
            _, rhythm = _dominant_period(sg, 384 / 22.0, 384 / 2.6)
        if fill >= 0.72:
            t = "solid"
        elif fill <= 0.06:
            t = "unmarked"
        elif rhythm >= 0.25 or fill < 0.62:
            t = "dashed"
        else:
            t = "solid"
        out.append({"type": t, "fill": fill, "rhythm": float(rhythm)})
    return out

def make_diagnostics(bg, band, paint, vp, lanes, calib, scale, prof, tprof, bnds):
    h, w = bg.shape[:2]
    img = bg.copy()
    edge = cv2.Canny(band, 50, 150)
    img[edge > 0] = (60, 220, 60)
    img[paint > 0] = (255, 255, 0)
    lm = LaneModel(lanes, calib, (h, w))
    img = lm.draw(img, alpha=0.18)
    for nm in lanes:
        for side, col in (("left", (0, 220, 255)), ("right", (0, 220, 255))):
            pts = np.asarray(lanes[nm][side], np.int32)
            if pts.shape[0] >= 2:
                cv2.polylines(img, [pts], False, col, 2, cv2.LINE_AA)
    vx, vy = int(round(vp[0])), int(round(vp[1]))
    if 0 <= vx < w:
        yy = int(_clamp(vy, 0, h - 1))
        cv2.drawMarker(img, (vx, yy), (0, 0, 255), cv2.MARKER_CROSS, 26, 2)
        cv2.line(img, (0, yy), (w, yy), (0, 0, 255), 1, cv2.LINE_AA)
        _label(img, "vanishing point / horizon", vx + 12, yy - 6, (0, 0, 255), 0.45)
    for Z in (10, 20, 30, 50, 75, 100):
        yv = calib.y_h + calib.k / float(Z)
        if calib.y_h + 2 < yv < h - 2:
            cv2.line(img, (0, int(yv)), (int(w * 0.06), int(yv)), (255, 255, 255), 1)
            _label(img, f"{Z} m", int(w * 0.065), int(yv) + 6, (40, 40, 40), 0.4)
    return img

# ==============================================================================
#  SECTION 8 - VIOLATION ENGINE
# ==============================================================================
class VehicleTrack:
    def __init__(self, tid, t, cls_name):
        self.tid = int(tid)
        self.cls = cls_name
        self.t0 = float(t)
        self.t = float(t)
        self.pts = deque(maxlen=400)
        self.spd = deque(maxlen=9)
        self.speed = 0.0
        self.peak_speed = 0.0
        self.lane = -1
        self.lane_stable = -1
        self.first_lane = -1
        self.cand_lane = -2
        self.cand_since = float(t)
        self.depart_t = None
        self.crossings = deque()
        self.lane_changes = []
        self.straddle_since = None
        self.straddle_max = 0.0
        self.stop_since = None
        self.over_since = None
        self.wrong_since = None
        self.logged = {}
        self.n = 0
        self.counted = False
        self.dist_m = 0.0
        self.max_lat = 0.0
        self.flags = []

    @property
    def age(self):
        return self.t - self.t0

    def can_log(self, vtype, t):
        sev, cooldown, repeat = VIOLATIONS.get(vtype, ("low", 10.0, True))
        last = self.logged.get(vtype)
        if last is None:
            return True
        if not repeat:
            return False
        return (t - last) >= cooldown

    def mark_logged(self, vtype, t):
        self.logged[vtype] = t

    def _window(self, t, secs, need_metric=True):
        lo = t - secs
        out = [p for p in self.pts if p[0] >= lo and
               (not need_metric or (np.isfinite(p[3]) and np.isfinite(p[4])))]
        return out

    def _update_speed(self, t):
        win = self._window(t, SPEED_WIN_SEC)
        if len(win) >= 2:
            dt = win[-1][0] - win[0][0]
            if dt > 0.08:
                d = math.hypot(win[-1][3] - win[0][3], win[-1][4] - win[0][4])
                v = d / dt * 3.6
                if 0.0 <= v < 400.0:
                    self.spd.append(v)
        if self.spd:
            self.speed = float(np.median(self.spd))
            if self.age > MIN_TRACK_SEC:
                self.peak_speed = max(self.peak_speed, self.speed)

    def _y_trend(self, t, secs=1.5):
        win = self._window(t, secs, need_metric=False)
        if len(win) < 3:
            return 0, 0.0
        ts = np.array([p[0] for p in win])
        ys = np.array([p[2] for p in win])
        if ts.max() - ts.min() < 0.2:
            return 0, 0.0
        try:
            slope = float(np.polyfit(ts - ts.min(), ys, 1)[0])
        except Exception:
            return 0, 0.0
        mw = [p for p in win if np.isfinite(p[3])]
        disp = 0.0
        if len(mw) >= 2:
            disp = math.hypot(mw[-1][3] - mw[0][3], mw[-1][4] - mw[0][4])
        return (1 if slope > 0 else -1) if abs(slope) > 1e-6 else 0, disp

    def update(self, t, box, lm, calib, cfg):
        x1, y1, x2, y2 = [float(v) for v in box]
        cx, cy = 0.5 * (x1 + x2), y2
        X, Z = calib.to_metric(cx, cy)
        prev = self.pts[-1] if self.pts else None
        self.pts.append((float(t), cx, cy, X, Z))
        self.t = float(t)
        self.n += 1
        if prev is not None and np.isfinite(X) and np.isfinite(prev[3]):
            self.dist_m += math.hypot(X - prev[3], Z - prev[4])
        self._update_speed(t)

        lane = lm.lane_at(cx, cy)
        self.lane = lane
        if self.first_lane < 0 and lane >= 0:
            self.first_lane = lane

        lat_mpp = calib.lat_m_per_px(cy)
        bw = max(2.0, x2 - x1)
        if np.isfinite(lat_mpp) and lat_mpp > 0:
            exp_px = _veh_width_m(self.cls) / lat_mpp
            eff_w = float(_clamp(exp_px, 0.5 * bw, bw))
        else:
            eff_w = bw
        fx1, fx2 = cx - eff_w * 0.5, cx + eff_w * 0.5

        out = []
        young = self.age < max(MIN_TRACK_SEC, cfg["min_track_sec"])

        if lane != self.cand_lane:
            self.cand_lane = lane
            self.cand_since = t
            if lane >= 0 and self.lane_stable >= 0 and lane != self.lane_stable:
                self.crossings.append(t)
        settled = (lane >= 0 and (t - self.cand_since) >= LANE_CHANGE_SETTLE_SEC)
        if settled and lane != self.lane_stable:
            if self.lane_stable >= 0:
                t_start = self.depart_t if self.depart_t is not None else self.cand_since
                dur = max(0.05, t - t_start)
                nb = abs(lane - self.lane_stable)
                crossed = list(range(min(lane, self.lane_stable) + 1,
                                     max(lane, self.lane_stable) + 1))
                types = [lm.boundary_types[i] if 0 <= i < len(lm.boundary_types)
                         else "unknown" for i in crossed]
                rec = {"t": t, "from": self.lane_stable, "to": lane, "dur": dur,
                       "n": nb, "types": types, "speed": self.speed}
                self.lane_changes.append(rec)
                if "solid" in types:
                    out.append(("Illegal Lane Change",
                                f"crossed a SOLID line "
                                f"({lm.name(self.lane_stable)} -> {lm.name(lane)})",
                                0.9))
                if nb >= 2:
                    out.append(("Unsafe Lane Change",
                                f"{nb} lanes at once in {dur:.1f}s", 0.8))
                elif dur < LANE_CHANGE_FAST_SEC:
                    out.append(("Unsafe Lane Change",
                                f"abrupt: {lm.name(self.lane_stable)} -> "
                                f"{lm.name(lane)} in {dur:.2f}s", 0.7))
            self.lane_stable = lane
            self.depart_t = None
            self.straddle_since = None
        if lane == self.lane_stable:
            self.depart_t = None
        elif self.depart_t is None:
            self.depart_t = t

        frac = 0.0
        if lane >= 0:
            xl, xr = lm.edges_at(lane, cy)
            if np.isfinite(xl) and np.isfinite(xr):
                over = max(xl - fx1, fx2 - xr, 0.0)
                frac = over / max(eff_w, 1e-6)
                lat, lw = lm.lateral_offset_m(lane, cx, cy)
                if np.isfinite(lat):
                    self.max_lat = max(self.max_lat, abs(lat))
        skip_2w = False
        if frac > cfg["straddle_overlap"] and not skip_2w:
            if self.straddle_since is None:
                self.straddle_since = t
            self.straddle_max = max(self.straddle_max, frac)
            held = t - self.straddle_since
            if held >= cfg["straddle_sec"] and not young:
                out.append(("Lane Straddling",
                            f"{frac * 100:.0f}% of the vehicle over the line "
                            f"for {held:.1f}s", _clamp(held / (2 * cfg["straddle_sec"])
                                                       * frac / 0.5, 0.3, 1.0)))
        else:
            self.straddle_since = None

        if lane >= 0 and lm.flow[lane] and lm.flow_trusted[lane] and not young:
            sgn, disp = self._y_trend(t, 1.6)
            if sgn and disp > 1.5 and sgn != lm.flow[lane]:
                if self.wrong_since is None:
                    self.wrong_since = t
                if (t - self.wrong_since) >= WRONGWAY_MIN_SEC and \
                        self.dist_m >= WRONGWAY_MIN_DISP_M:
                    out.append(("Wrong-Way",
                                f"counter-flow in {lm.name(lane)} "
                                f"({self.dist_m:.0f} m against traffic)", 0.95))
            else:
                self.wrong_since = None

        if self.speed > cfg["speed_limit"] and not young:
            if self.over_since is None:
                self.over_since = t
            if (t - self.over_since) >= 0.4:
                out.append(("Overspeeding",
                            f"{self.speed:.0f} km/h in a {cfg['speed_limit']:.0f} "
                            f"km/h zone", _clamp((self.speed - cfg["speed_limit"]) /
                                                 max(10.0, 0.25 * cfg["speed_limit"]),
                                                 0.4, 1.0)))
        else:
            self.over_since = None

        win = self._window(t, cfg["stop_sec"])
        if len(win) >= 3 and (win[-1][0] - win[0][0]) >= cfg["stop_sec"] * 0.85:
            Xs = np.array([p[3] for p in win])
            Zs = np.array([p[4] for p in win])
            spread = math.hypot(Xs.max() - Xs.min(), Zs.max() - Zs.min())
            if spread < STOP_DISP_M:
                if self.stop_since is None:
                    self.stop_since = win[0][0]
                out.append(("Unauthorized Parking",
                            f"moved {spread:.1f} m in "
                            f"{t - self.stop_since:.0f}s", 0.85))
            else:
                self.stop_since = None
        elif len(win) < 3:
            pw = self._window(t, cfg["stop_sec"], need_metric=False)
            if len(pw) >= 3 and (pw[-1][0] - pw[0][0]) >= cfg["stop_sec"] * 0.85 and \
                    np.isfinite(lat_mpp) and lat_mpp > 0:
                xs = np.array([p[1] for p in pw])
                ys = np.array([p[2] for p in pw])
                spread_px = math.hypot(xs.max() - xs.min(), ys.max() - ys.min())
                if spread_px * lat_mpp < STOP_DISP_M:
                    if self.stop_since is None:
                        self.stop_since = pw[0][0]
                    out.append(("Unauthorized Parking",
                                f"stationary for {t - self.stop_since:.0f}s", 0.6))

        self.flags = [o[0] for o in out]
        return out

class ViolationEngine:
    def __init__(self, lm, calib, cfg):
        self.lm = lm
        self.calib = calib
        self.cfg = cfg
        self.tracks = {}
        self.retired = {}
        self.log = []
        self.lane_change_log = []

    def frame(self, t, dets):
        live = []
        for tid, cls_name, box in dets:
            tid = int(tid)
            tr = self.tracks.get(tid)
            if tr is None:
                tr = VehicleTrack(tid, t, cls_name)
                self.tracks[tid] = tr
            tr.cls = cls_name
            live.append((tid, tr, box))

        result = {}
        for tid, tr, box in live:
            result[tid] = (tr, tr.update(t, box, self.lm, self.calib, self.cfg))

        stale = [k for k, v in self.tracks.items() if (t - v.t) > 4.0]
        for k in stale:
            self.retired[k] = self.tracks.pop(k)
        return result

    def commit(self, t, tr, flags, frame_bgr, box, save_evidence=True):
        added = []
        for vtype, detail, conf in flags:
            if not tr.can_log(vtype, t):
                continue
            tr.mark_logged(vtype, t)
            sev = VIOLATIONS.get(vtype, ("low", 10, True))[0]
            ev = None
            if save_evidence and frame_bgr is not None:
                ev = save_evidence_crop(frame_bgr, box, tr, vtype, t)
            entry = {
                "id": tr.tid, "type": vtype, "class": tr.cls, "timestamp": float(t),
                "severity": sev, "confidence": round(float(conf), 2),
                "lane": self.lm.name(tr.lane) if tr.lane >= 0 else
                        (self.lm.name(tr.lane_stable) if tr.lane_stable >= 0 else ""),
                "details": detail,
                "speed": round(float(tr.speed), 1),
                "evidence": ev,
            }
            self.log.append(entry)
            added.append(entry)
        return added

    def all_tracks(self):
        d = dict(self.retired)
        d.update(self.tracks)
        return d

_EVID_N = [0]

def save_evidence_crop(frame, box, tr, vtype, t, max_side=360):
    try:
        h, w = frame.shape[:2]
        x1, y1, x2, y2 = [int(round(v)) for v in box]
        pad_x = int(0.35 * max(24, x2 - x1))
        pad_y = int(0.45 * max(24, y2 - y1))
        cx1 = int(_clamp(x1 - pad_x, 0, w - 2))
        cy1 = int(_clamp(y1 - pad_y, 0, h - 2))
        cx2 = int(_clamp(x2 + pad_x, cx1 + 2, w))
        cy2 = int(_clamp(y2 + pad_y, cy1 + 2, h))
        crop = frame[cy1:cy2, cx1:cx2].copy()
        if crop.size == 0:
            return None
        col = VIOL_BOX_COLOR.get(vtype, (0, 0, 255))
        cv2.rectangle(crop, (x1 - cx1, y1 - cy1), (x2 - cx1, y2 - cy1), col, 2)
        sc = max_side / float(max(crop.shape[:2]))
        if sc < 1.0:
            crop = cv2.resize(crop, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
        crop = np.vstack([crop, np.zeros((26, crop.shape[1], 3), np.uint8)])
        _label(crop, f"ID{tr.tid} {tr.cls} | {vtype} | {_ts(t)}",
               3, crop.shape[0] - 5, col, 0.42)
        _EVID_N[0] += 1
        safe = vtype.lower().replace(" ", "_").replace("-", "_")
        fn = os.path.join(EVID_DIR, f"v{_EVID_N[0]:05d}_id{tr.tid}_{safe}.jpg")
        cv2.imwrite(fn, crop, [int(cv2.IMWRITE_JPEG_QUALITY), 88])
        return fn
    except Exception:
        return None

def reset_evidence():
    _EVID_N[0] = 0
    try:
        for f in os.listdir(EVID_DIR):
            if f.endswith(".jpg"):
                os.remove(os.path.join(EVID_DIR, f))
    except Exception:
        pass

# ==============================================================================
#  SECTION 9 - ANALYSIS PASS
# ==============================================================================
PROC_MAX_W = 1600

def _los_from_speed(v_kmh, free_flow=None):
    if v_kmh is None or not np.isfinite(v_kmh) or v_kmh <= 0:
        return "-", "#6b7280"
    ff = free_flow if (free_flow and free_flow > 5) else 60.0
    r = v_kmh / ff
    if r >= 0.90:
        return "A - free flow", "#22c55e"
    if r >= 0.75:
        return "B - reasonably free", "#4ade80"
    if r >= 0.60:
        return "C - stable", "#a3e635"
    if r >= 0.45:
        return "D - approaching unstable", "#eab308"
    if r >= 0.30:
        return "E - unstable / at capacity", "#f97316"
    return "F - forced flow / jam", "#ef4444"

def analyze_video(video_path, calib, opts, progress=None):
    log = []

    def say(msg):
        log.append(msg)
        if progress is not None:
            try:
                progress(0, desc=msg)
            except Exception:
                pass

    model, minfo = get_model(opts.get("model_path") or MODEL_PATH)
    if model is None:
        return None, "Model unavailable: " + minfo.get("msg", "unknown error")

    meta = video_meta(video_path)
    if not meta["ok"]:
        return None, "Could not open the video."
    W0, H0, fps, nframes = meta["w"], meta["h"], meta["fps"], meta["frames"]

    s = min(1.0, PROC_MAX_W / float(W0))
    W, H = int(round(W0 * s)), int(round(H0 * s))
    pcal = PerspectiveCalib.from_dict(calib["perspective"])
    if abs(pcal.frame_w - W) > 2:
        pcal = pcal.rescaled(W / float(pcal.frame_w))
    lanes = _scaled_lanes(calib["lanes"], W / float(calib.get("frame_w", W0) or W0))
    lm = LaneModel(lanes, pcal, (H, W))
    if not lm.names:
        return None, "No lanes in the calibration."

    cfg = {
        "speed_limit": float(opts.get("speed_limit", DEFAULT_SPEED_LIMIT)),
        "stop_sec": float(opts.get("stop_sec", DEFAULT_STOP_SEC)),
        "straddle_sec": float(opts.get("straddle_sec", STRADDLE_SEC)),
        "straddle_overlap": float(opts.get("straddle_overlap", STRADDLE_OVERLAP)),
        "ignore_2w": False,
        "min_track_sec": MIN_TRACK_SEC,
    }
    enabled = set(opts.get("enabled_violations") or VIOLATIONS.keys())
    eng = ViolationEngine(lm, pcal, cfg)
    reset_evidence()

    stride = 2
    conf_th = 0.25
    iou_th = 0.50
    want_video = True
    save_evid = True
    line_y = int(0.72 * H)

    ensure_tracker_cfg()
    dev = get_device()

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, "Could not open the video for reading."
    writer = None
    if want_video:
        for fourcc in ("mp4v", "avc1", "XVID"):
            writer = cv2.VideoWriter(OUT_VIDEO, _fourcc(fourcc),
                                     max(1.0, fps / stride), (W, H))
            if writer.isOpened():
                break
            writer.release()
            writer = None
        if writer is None:
            say("Video writer unavailable - continuing without an output video.")
            want_video = False

    counted = {}
    cls_counts = Counter()
    lane_counts = Counter()
    lane_pcu = defaultdict(float)
    speed_samples = defaultdict(list)
    all_speeds = []
    occupancy = defaultdict(int)
    per_frame = []
    prev_side = {}
    frames_done = 0
    idx = -1
    t0 = time.time()
    say("Analysing frames...")

    try:
        while True:
            ok, raw = cap.read()
            if not ok:
                break
            idx += 1
            if idx % stride:
                continue
            frame = cv2.resize(raw, (W, H), interpolation=cv2.INTER_AREA) \
                if s < 1.0 else raw
            t = idx / max(fps, 1e-6)

            try:
                res = model.track(frame, persist=True, conf=conf_th, iou=iou_th,
                                  tracker=TRACKER_CFG, verbose=False,
                                  device=dev, classes=None)
            except Exception as e:
                log.append(f"frame {idx}: {e}")
                res = None

            dets = []
            if res:
                r = res[0]
                names = r.names if hasattr(r, "names") else {}
                if r.boxes is not None and r.boxes.id is not None:
                    xyxy = r.boxes.xyxy.cpu().numpy()
                    ids = r.boxes.id.cpu().numpy().astype(int)
                    clss = r.boxes.cls.cpu().numpy().astype(int)
                    for b, tid, ci in zip(xyxy, ids, clss):
                        nm = str(names.get(int(ci), ci))
                        if not _is_vehicle(nm):
                            continue
                        dets.append((int(tid), nm, (float(b[0]), float(b[1]),
                                                    float(b[2]), float(b[3]))))

            out = eng.frame(t, dets)
            annot = frame.copy() if want_video else None
            if annot is not None:
                annot = lm.draw(annot)
                cv2.line(annot, (0, line_y), (W, line_y), (255, 255, 255), 1, cv2.LINE_AA)

            n_live = 0
            frame_speeds = []
            for tid, cls_name, box in dets:
                tr, flags = out[tid]
                n_live += 1
                x1, y1, x2, y2 = box
                cx, cy = 0.5 * (x1 + x2), y2

                side = 1 if cy >= line_y else 0
                was = prev_side.get(tid)
                prev_side[tid] = side
                if was is not None and was != side and tid not in counted:
                    ln = tr.lane if tr.lane >= 0 else tr.lane_stable
                    counted[tid] = {"lane": ln, "cls": cls_name, "t": t,
                                    "dir": "down" if side == 1 else "up"}
                    tr.counted = True
                    cls_counts[cls_name] += 1
                    key = lm.name(ln) if ln >= 0 else "off-lane"
                    lane_counts[key] += 1
                    lane_pcu[key] += _pcu(cls_name)

                if tr.lane >= 0:
                    occupancy[tr.lane] += 1
                    if tr.speed > 0.5 and tr.age > MIN_TRACK_SEC:
                        speed_samples[tr.lane].append(tr.speed)
                        all_speeds.append(tr.speed)
                        frame_speeds.append(tr.speed)

                keep = [f for f in flags if f[0] in enabled]
                new = eng.commit(t, tr, keep, frame if save_evid else None, box,
                                 save_evidence=save_evid)
                if annot is not None:
                    _draw_vehicle(annot, box, tr, keep, new)

            if annot is not None:
                writer.write(annot)

            per_frame.append((t, n_live,
                              float(np.mean(frame_speeds)) if frame_speeds else np.nan))
            frames_done += 1
            if progress is not None and frames_done % 25 == 0 and nframes:
                frac = _clamp(idx / float(nframes), 0.0, 1.0)
                try:
                    progress(frac, desc=f"Analysing {int(frac * 100)}%  "
                                        f"({len(eng.log)} violations)")
                except Exception:
                    pass
    finally:
        cap.release()
        if writer is not None:
            writer.release()

    dur = max(1e-6, (idx + 1) / max(fps, 1e-6))
    elapsed = time.time() - t0
    say(f"Analysed {frames_done} frames in {elapsed:.0f}s.")

    tracks = eng.all_tracks()
    lane_rows = []
    for i, nm in enumerate(lm.names):
        sp = speed_samples.get(i, [])
        vol = lane_counts.get(nm, 0)
        v_mean = float(np.mean(sp)) if sp else float("nan")
        v_p85 = float(np.percentile(sp, 85)) if len(sp) >= 8 else float("nan")
        flow = vol * 3600.0 / dur
        los, los_col = _los_from_speed(v_mean, cfg["speed_limit"])
        lane_rows.append({
            "Lane": nm,
            "Type": "Shoulder" if lm.kind[i] != "travel" else "Travel",
            "Left line": lm.boundary_types[i] if i < len(lm.boundary_types) else "-",
            "Right line": lm.boundary_types[i + 1] if i + 1 < len(lm.boundary_types) else "-",
            "Direction": {1: "Toward camera", -1: "Away", 0: "Unknown"}[lm.flow[i]],
            "Vehicles": vol,
            "PCU": round(lane_pcu.get(nm, 0.0), 1),
            "Flow (veh/h)": round(flow, 1),
            "PCU/h": round(lane_pcu.get(nm, 0.0) * 3600.0 / dur, 1),
            "Avg speed (km/h)": round(v_mean, 1) if np.isfinite(v_mean) else "-",
            "85th %ile (km/h)": round(v_p85, 1) if np.isfinite(v_p85) else "-",
            "Occupancy (%)": round(100.0 * occupancy.get(i, 0) / max(1, frames_done), 1),
            "LOS": los,
            "_los_color": los_col,
        })

    v_all = float(np.mean(all_speeds)) if all_speeds else float("nan")
    los, los_col = _los_from_speed(v_all, cfg["speed_limit"])
    total = sum(cls_counts.values())
    results = {
        "ok": True,
        "video": video_path,
        "duration": dur,
        "fps": fps,
        "frames_processed": frames_done,
        "stride": stride,
        "width": W0, "height": H0,
        "elapsed": elapsed,
        "device": device_label(),
        "model": minfo,
        "counts": dict(cls_counts),
        "total": total,
        "pcu_total": round(sum(_pcu(c) * n for c, n in cls_counts.items()), 1),
        "lane_rows": lane_rows,
        "lane_counts": dict(lane_counts),
        "violations": eng.log,
        "lane_changes": [dict(lc, id=tid, cls=tr.cls)
                         for tid, tr in tracks.items() for lc in tr.lane_changes],
        "tracks": {tid: {"class": tr.cls, "lane": lm.name(tr.lane_stable),
                         "frames": tr.n, "dist_m": round(tr.dist_m, 1),
                         "avg_speed": round(tr.speed, 1),
                         "peak_speed": round(tr.peak_speed, 1),
                         "max_offset_m": round(tr.max_lat, 2),
                         "flags": sorted(set(tr.logged.keys())),
                         "counted": tr.counted}
                   for tid, tr in tracks.items()},
        "per_frame": per_frame,
        "avg_speed": round(v_all, 1) if np.isfinite(v_all) else None,
        "los": los, "los_color": los_col,
        "calibration": calib,
        "lane_model": lm,
        "cfg": cfg,
        "enabled": sorted(enabled),
        "log": log,
        "out_video": OUT_VIDEO if want_video and os.path.exists(OUT_VIDEO) else None,
        "created": _now_str(),
    }
    results["violations_by_type"] = dict(Counter(v["type"] for v in eng.log))
    results["violations_by_severity"] = dict(Counter(v["severity"] for v in eng.log))
    results["violation_rate"] = round(100.0 * len({v["id"] for v in eng.log}) /
                                      max(1, len(tracks)), 1)
    if want_video and results["out_video"]:
        results["out_video"] = _transcode_h264(results["out_video"])
    return results, None

def _scaled_lanes(lanes, s):
    if abs(s - 1.0) < 1e-6:
        return lanes
    out = {}
    for nm, L in lanes.items():
        d = dict(L)
        for key in ("polygon", "left", "right"):
            if L.get(key):
                d[key] = [[float(p[0]) * s, float(p[1]) * s] for p in L[key]]
        if d.get("polygon"):
            d["polygon"] = [[int(round(p[0])), int(round(p[1]))] for p in d["polygon"]]
        out[nm] = d
    return out

def _draw_vehicle(img, box, tr, flags, newly):
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    if flags:
        worst = min(flags, key=lambda f: SEV_ORDER.get(
            VIOLATIONS.get(f[0], ("low",))[0], 3))[0]
        col = VIOL_BOX_COLOR.get(worst, (0, 0, 255))
        th = 3 if newly else 2
    else:
        col, th = _cls_color(tr.cls), 2
    cv2.rectangle(img, (x1, y1), (x2, y2), col, th)
    tag = f"{tr.tid} {tr.cls}"
    if tr.speed > 1:
        tag += f" {tr.speed:.0f}km/h"
    _label(img, tag, x1, y1 - 2, col, 0.42)
    for j, (vt, detail, cf) in enumerate(flags[:3]):
        _label(img, vt, x1, y2 + 16 + j * 16,
               VIOL_BOX_COLOR.get(vt, (0, 0, 255)), 0.40)
    if newly:
        cv2.rectangle(img, (x1 - 3, y1 - 3), (x2 + 3, y2 + 3), (255, 255, 255), 1)

def _now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def _transcode_h264(path):
    out = path.replace(".mp4", "_h264.mp4")
    try:
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path,
                            "-c:v", "libx264", "-preset", "veryfast", "-crf", "26",
                            "-pix_fmt", "yuv420p", "-movflags", "+faststart", out],
                           capture_output=True, timeout=3600)
        if r.returncode == 0 and os.path.exists(out) and os.path.getsize(out) > 1024:
            return out
    except Exception:
        pass
    return path

# ==============================================================================
#  SECTION 10 - REPORTING
# ==============================================================================
def _b64_img(path, max_w=None):
    try:
        img = cv2.imread(path)
        if img is None:
            return ""
        if max_w and img.shape[1] > max_w:
            sc = max_w / float(img.shape[1])
            img = cv2.resize(img, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
        ok, buf = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), 82])
        if not ok:
            return ""
        return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode()
    except Exception:
        return ""

def _xl_style(ws, headers, widths=None, freeze="A2"):
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    hdr_fill = PatternFill("solid", fgColor="1F2937")
    hdr_font = Font(bold=True, color="FFFFFF", size=10)
    thin = Side(style="thin", color="D1D5DB")
    for c, name in enumerate(headers, start=1):
        cell = ws.cell(row=1, column=c, value=name)
        cell.fill = hdr_fill
        cell.font = hdr_font
        cell.alignment = Alignment(horizontal="center", vertical="center",
                                   wrap_text=True)
        cell.border = Border(bottom=thin)
        ws.column_dimensions[cell.column_letter].width = \
            (widths[c - 1] if widths and c - 1 < len(widths) else
             max(11, min(42, len(str(name)) + 6)))
    ws.freeze_panes = freeze
    return ws

def _xl_rows(ws, rows, start=2, sev_col=None):
    from openpyxl.styles import Font, PatternFill, Alignment
    fills = {"critical": "FEE2E2", "high": "FFEDD5", "medium": "FEF9C3", "low": "F3F4F6"}
    for i, row in enumerate(rows):
        for j, v in enumerate(row, start=1):
            cell = ws.cell(row=start + i, column=j,
                           value=(v if not isinstance(v, (np.floating, np.integer))
                                  else v.item()))
            cell.alignment = Alignment(vertical="center", wrap_text=(j == len(row)))
            cell.font = Font(size=10)
        if sev_col is not None and sev_col <= len(row):
            key = str(row[sev_col - 1]).lower()
            if key in fills:
                for j in range(1, len(row) + 1):
                    ws.cell(row=start + i, column=j).fill = \
                        PatternFill("solid", fgColor=fills[key])
    return ws

def build_excel(res, path=XLSX_PATH, thumbs=True):
    try:
        from openpyxl import Workbook
        from openpyxl.drawing.image import Image as XLImage
        from openpyxl.styles import Font, Alignment
        from openpyxl.utils import get_column_letter
    except Exception as e:
        return None, f"openpyxl unavailable: {e}"

    cal = res.get("calibration") or {}
    pers = cal.get("perspective") or {}
    wb = Workbook()

    ws = wb.active
    ws.title = "Summary"
    ws["A1"] = f"{APP_NAME} {APP_VER} - Traffic Analysis Report"
    ws["A1"].font = Font(bold=True, size=15)
    ws.merge_cells("A1:D1")
    ws["A2"] = f"Generated {res.get('created', '')}"
    ws["A2"].font = Font(italic=True, size=9, color="6B7280")
    approx = "approximate" in str(pers.get("scale_src", "")).lower() or \
             str(pers.get("scale_src", "")).startswith("assumed")
    pairs = [
        ("Source video", os.path.basename(res.get("video", "") or "")),
        ("Resolution", f"{res.get('width')}x{res.get('height')}"),
        ("Duration", _ts(res.get("duration", 0))),
        ("Frames analysed", res.get("frames_processed")),
        ("Frame stride", res.get("stride")),
        ("Processing time", f"{res.get('elapsed', 0):.0f} s"),
        ("Compute device", res.get("device")),
        ("Detector", (res.get("model") or {}).get("msg", "")),
        ("", ""),
        ("Total vehicles counted", res.get("total")),
        ("Total PCU", res.get("pcu_total")),
        ("Mean speed (km/h)", res.get("avg_speed") if res.get("avg_speed") else "-"),
        ("Level of service", res.get("los")),
        ("Unique tracks", len(res.get("tracks") or {})),
        ("Vehicles with >=1 violation", f"{res.get('violation_rate')} %"),
        ("Total violations", len(res.get("violations") or [])),
        ("", ""),
        ("Lanes detected", cal.get("n_lanes")),
        ("Lane engine", cal.get("engine")),
        ("Vanishing point source", cal.get("vp_source")),
        ("Geometry confidence", f"{100 * float(cal.get('geometry_conf', 0)):.0f} %"),
        ("Longitudinal scale", pers.get("scale_src", "-")),
        ("Speed accuracy", "APPROXIMATE - no dash rhythm found, focal length assumed"
                           if approx else "Metric, pinned by dashed-marking period"),
        ("Assumed lane width (m)", pers.get("lane_w_m")),
    ]
    r = 4
    for k, v in pairs:
        if k:
            ws.cell(row=r, column=1, value=k).font = Font(bold=True, size=10)
            ws.cell(row=r, column=2, value=v).font = Font(size=10)
        r += 1
    ws.column_dimensions["A"].width = 32
    ws.column_dimensions["B"].width = 62

    br = 4
    ws.cell(row=br, column=4, value="Violations by type").font = Font(bold=True)
    for name, n in sorted((res.get("violations_by_type") or {}).items(),
                          key=lambda kv: -kv[1]):
        br += 1
        ws.cell(row=br, column=4, value=name).font = Font(size=10)
        ws.cell(row=br, column=5, value=n)
    br += 2
    ws.cell(row=br, column=4, value="Vehicle mix").font = Font(bold=True)
    for name, n in sorted((res.get("counts") or {}).items(), key=lambda kv: -kv[1]):
        br += 1
        ws.cell(row=br, column=4, value=name).font = Font(size=10)
        ws.cell(row=br, column=5, value=n)
    ws.column_dimensions["D"].width = 26
    ws.column_dimensions["E"].width = 10

    vs = wb.create_sheet("Violations")
    headers = ["#", "Time", "Track", "Vehicle", "Violation", "Severity",
               "Conf.", "Lane", "Speed (km/h)", "Evidence", "Details"]
    _xl_style(vs, headers, widths=[5, 9, 8, 13, 20, 10, 7, 14, 12, 26, 52])
    viols = sorted(res.get("violations") or [],
                   key=lambda v: (SEV_ORDER.get(v["severity"], 3), v["timestamp"]))
    rows = [[i + 1, _ts(v["timestamp"]), v["id"], v["class"], v["type"],
             SEV_LABELS.get(v["severity"], v["severity"]), v["confidence"],
             v["lane"], v["speed"], "", v["details"]]
            for i, v in enumerate(viols)]
    _xl_rows(vs, rows, sev_col=6)
    if thumbs:
        THUMB_H = 74
        col = get_column_letter(headers.index("Evidence") + 1)
        for i, v in enumerate(viols):
            p = v.get("evidence")
            if not p or not os.path.exists(p):
                continue
            try:
                im = XLImage(p)
                sc = THUMB_H / float(im.height or THUMB_H)
                im.height = int(im.height * sc)
                im.width = int(im.width * sc)
                vs.add_image(im, f"{col}{i + 2}")
                vs.row_dimensions[i + 2].height = THUMB_H * 0.78
            except Exception:
                vs.cell(row=i + 2, column=headers.index("Evidence") + 1,
                        value=os.path.basename(p))
    if not rows:
        vs.cell(row=2, column=1, value="No violations detected.").font = Font(italic=True)

    ls = wb.create_sheet("Lane Analysis")
    lane_rows = res.get("lane_rows") or []
    if lane_rows:
        lh = [k for k in lane_rows[0].keys() if not k.startswith("_")]
        _xl_style(ls, lh, widths=[14, 10, 11, 11, 15, 10, 8, 13, 10, 15, 15, 13, 24])
        _xl_rows(ls, [[r.get(k) for k in lh] for r in lane_rows])

    cs = wb.create_sheet("Vehicle Classes")
    _xl_style(cs, ["Class", "Count", "Share (%)", "PCU each", "PCU total"],
              widths=[20, 10, 12, 11, 12])
    tot = max(1, res.get("total", 0))
    _xl_rows(cs, [[k, n, round(100.0 * n / tot, 1), _pcu(k), round(_pcu(k) * n, 1)]
                  for k, n in sorted((res.get("counts") or {}).items(),
                                     key=lambda kv: -kv[1])])

    ts_ = wb.create_sheet("Tracks")
    _xl_style(ts_, ["Track", "Vehicle", "Lane", "Frames", "Distance (m)",
                    "Avg speed", "Peak speed", "Max offset from centre (m)",
                    "Counted", "Violations"],
              widths=[8, 13, 14, 9, 13, 11, 11, 24, 10, 34])
    _xl_rows(ts_, [[tid, d["class"], d["lane"], d["frames"], d["dist_m"],
                    d["avg_speed"], d["peak_speed"], d["max_offset_m"],
                    "yes" if d["counted"] else "no", ", ".join(d["flags"])]
                   for tid, d in sorted((res.get("tracks") or {}).items())])

    lc = wb.create_sheet("Lane Changes")
    _xl_style(lc, ["Time", "Track", "Vehicle", "From", "To", "Lanes crossed",
                   "Duration (s)", "Speed (km/h)", "Lines crossed"],
              widths=[9, 8, 13, 10, 10, 13, 13, 13, 24])
    lm = res.get("lane_model")
    _xl_rows(lc, [[_ts(c["t"]), c["id"], c["cls"],
                   lm.name(c["from"]) if lm else c["from"],
                   lm.name(c["to"]) if lm else c["to"], c["n"],
                   round(c["dur"], 2), round(c["speed"], 1), ", ".join(c["types"])]
                  for c in sorted(res.get("lane_changes") or [],
                                  key=lambda c: c["t"])])

    ks = wb.create_sheet("Calibration")
    ks["A1"] = "Geometry & metric calibration"
    ks["A1"].font = Font(bold=True, size=13)
    row = 3
    try:
        desc = PerspectiveCalib.from_dict(pers).describe()
    except Exception:
        desc = {}
    for k, v in desc.items():
        ks.cell(row=row, column=1, value=k).font = Font(bold=True, size=10)
        ks.cell(row=row, column=2, value=str(v)).font = Font(size=10)
        row += 1
    row += 1
    ks.cell(row=row, column=1, value="Boundary lines (left to right)").font = Font(bold=True)
    row += 1
    for i, (bt, bf) in enumerate(zip(cal.get("boundary_types") or [],
                                     cal.get("boundary_fill") or [])):
        ks.cell(row=row, column=1, value=f"Line {i + 1}").font = Font(size=10)
        ks.cell(row=row, column=2, value=f"{bt}  (paint fill {bf:.0%})").font = Font(size=10)
        row += 1
    row += 1
    ks.cell(row=row, column=1, value="Calibration log").font = Font(bold=True)
    row += 1
    rep = (cal.get("report") or {})
    for line in (rep.get("steps") or []) + (rep.get("notes") or []):
        c = ks.cell(row=row, column=1, value=str(line))
        c.font = Font(size=10)
        c.alignment = Alignment(wrap_text=True)
        ks.merge_cells(start_row=row, start_column=1, end_row=row, end_column=4)
        row += 1
    ks.column_dimensions["A"].width = 34
    ks.column_dimensions["B"].width = 46

    try:
        wb.save(path)
    except Exception as e:
        alt = os.path.join(SAVE_DIR, f"traffic_report_{int(time.time())}.xlsx")
        try:
            wb.save(alt)
            return alt, f"Saved to {alt} (original path was locked: {e})"
        except Exception as e2:
            return None, f"Could not save the workbook: {e2}"
    return path, None

# ------------------------------------------------------------------ HTML dashboard
_CSS = """
:root { --bg: #f8fafc; --text: #0f172a; --card-bg: #ffffff; --border: #e2e8f0; --th-bg: #f1f5f9; --pill-text: #ffffff; }
body.dark { --bg: #0b1220; --text: #e5e7eb; --card-bg: #121c2f; --border: #1f2c44; --th-bg: #18243b; --pill-text: #000000; }
*{box-sizing:border-box}
body{margin:0;font:14px/1.55 -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;background:var(--bg);color:var(--text); transition: background 0.2s ease, color 0.2s ease;}
.wrap{max-width:1180px;margin:0 auto;padding:22px}
h1{font-size:22px;margin:0 0 2px}
h2{font-size:15px;margin:26px 0 10px;letter-spacing:.06em;text-transform:uppercase;color:#93a3b8;font-weight:600}
.sub{color:#8296ae;font-size:12px;margin-bottom:18px}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(158px,1fr));gap:11px}
.card{background:var(--card-bg);border:1px solid var(--border);border-radius:11px;padding:13px 14px}
.card .k{font-size:11px;color:#8296ae;text-transform:uppercase;letter-spacing:.06em}
.card .v{font-size:23px;font-weight:650;margin-top:5px}
.card .n{font-size:11px;color:#6b7f96;margin-top:3px}
.table-container { overflow-x: auto; margin-bottom: 20px;}
table{width:100%;border-collapse:collapse;background:var(--card-bg);border-radius:11px;overflow:hidden;font-size:13px; white-space:nowrap; border:1px solid var(--border);}
th{background:var(--th-bg);text-align:left;padding:9px 11px;font-size:11px;text-transform:uppercase;letter-spacing:.05em;color:#93a3b8}
td{padding:8px 11px;border-top:1px solid var(--border)}
tr:hover td{background:var(--th-bg)}
.pill{display:inline-block;padding:2px 9px;border-radius:999px;font-size:11px;font-weight:600;color:var(--pill-text)}
.bar{height:7px;background:var(--border);border-radius:4px;overflow:hidden;min-width:70px}
.bar>i{display:block;height:100%;background:#38bdf8}
.gal{display:grid;grid-template-columns:repeat(auto-fill,minmax(196px,1fr));gap:11px}
.ev{background:var(--card-bg);border:1px solid var(--border);border-radius:10px;overflow:hidden}
.ev img{width:100%;display:block}.ev .m{padding:7px 9px;font-size:11px;color:#9fb2c8}
.tabs{display:flex;gap:6px;margin:18px 0 4px;flex-wrap:wrap}
.tab{padding:7px 15px;border-radius:8px;background:var(--card-bg);border:1px solid var(--border);cursor:pointer;font-size:13px;color:var(--text)}
.tab.on{background:#1d4ed8;border-color:#1d4ed8;color:#fff}
.pane{display:none;animation:f .18s ease}.pane.on{display:block}
@keyframes f{from{opacity:0}to{opacity:1}}
.note{background:var(--card-bg);border:1px solid var(--border); border-left:4px solid #38bdf8;padding:9px 13px;border-radius:4px;margin:7px 0;font-size:12.5px;color:var(--text)}
.warn{border-left-color:#f59e0b}
img.diag{width:100%;border-radius:11px;border:1px solid var(--border)}
.mono{font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px}
.theme-btn {background: var(--card-bg); color: var(--text); border: 1px solid var(--border); padding: 5px 12px; border-radius: 6px; cursor: pointer; float: right; margin-top: 10px; font-weight: bold;}
"""

_JS = """
function tab(n){document.querySelectorAll('.tab').forEach(function(t,i){
t.classList.toggle('on',i===n)});document.querySelectorAll('.pane').forEach(
function(p,i){p.classList.toggle('on',i===n)});}
"""

def _esc(s):
    return (str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))

def _card(k, v, n=""):
    return (f"<div class='card'><div class='k'>{_esc(k)}</div>"
            f"<div class='v'>{_esc(v)}</div>"
            f"<div class='n'>{_esc(n)}</div></div>")

def _table(headers, rows):
    h = "".join(f"<th>{_esc(x)}</th>" for x in headers)
    b = "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>" for r in rows)
    return f"<div class='table-container'><table><thead><tr>{h}</tr></thead><tbody>{b}</tbody></table></div>"

def build_dashboard(res, max_evidence=48):
    cal = res.get("calibration") or {}
    pers = cal.get("perspective") or {}
    viols = res.get("violations") or []
    lm = res.get("lane_model")
    n_tracks = len(res.get("tracks") or {})

    cards = [
        _card("Vehicles", res.get("total", 0),
              f"{res.get('pcu_total', 0)} PCU  ·  {n_tracks} tracks"),
        _card("Violations", len(viols),
              f"{res.get('violation_rate', 0)}% of vehicles"),
        _card("Mean speed", f"{res.get('avg_speed') or '--'} km/h",
              res.get("los", "")),
        _card("Lanes", cal.get("n_lanes", len(lm.names) if lm else 0),
              f"{lm.n_travel() if lm else '-'} travel lanes"),
        _card("Duration", _ts(res.get("duration", 0)),
              f"{res.get('frames_processed', 0)} frames analysed"),
        _card("Geometry conf.", f"{100 * float(cal.get('geometry_conf', 0)):.0f}%",
              cal.get("vp_source", "")),
    ]
    sev = res.get("violations_by_severity") or {}
    sev_rows = [[f"<span class='pill' style='background:{SEV_COLORS.get(k, '#6b7280')}'>"
                 f"{SEV_LABELS.get(k, k)}</span>", n,
                 f"<div class='bar'><i style='width:{100 * n / max(1, len(viols)):.0f}%;"
                 f"background:{SEV_COLORS.get(k, '#6b7280')}'></i></div>"]
                for k, n in sorted(sev.items(), key=lambda kv: SEV_ORDER.get(kv[0], 3))]
    vt = res.get("violations_by_type") or {}
    mx = max(vt.values()) if vt else 1
    type_rows = [[_esc(k), n,
                  f"<div class='bar'><i style='width:{100 * n / mx:.0f}%'></i></div>"]
                 for k, n in sorted(vt.items(), key=lambda kv: -kv[1])]
    mix = res.get("counts") or {}
    tot = max(1, res.get("total", 0))
    mix_rows = [[_esc(k), n, f"{100 * n / tot:.1f}%", _pcu(k)]
                for k, n in sorted(mix.items(), key=lambda kv: -kv[1])]

    overview = ("<div class='cards'>" + "".join(cards) + "</div>"
                + "<h2>Severity</h2>" + (_table(["Severity", "Count", ""], sev_rows)
                                         if sev_rows else "<div class='note'>No violations "
                                                          "were detected in this clip.</div>")
                + "<h2>Violation types</h2>"
                + (_table(["Type", "Count", ""], type_rows) if type_rows else "")
                + "<h2>Vehicle mix</h2>" + _table(["Class", "Count", "Share", "PCU"], mix_rows))

    vrows = []
    for v in sorted(viols, key=lambda v: (SEV_ORDER.get(v["severity"], 3), v["timestamp"])):
        vrows.append([
            f"<span class='mono'>{_ts(v['timestamp'])}</span>", v["id"], _esc(v["class"]),
            f"<span class='pill' style='background:{SEV_COLORS.get(v['severity'], '#6b7280')}'>"
            f"{_esc(v['type'])}</span>",
            _esc(v["lane"]), v["speed"], f"{v['confidence']:.2f}", _esc(v["details"])])
    vtab = _table(["Time", "Track", "Vehicle", "Violation", "Lane", "km/h",
                   "Conf.", "Detail"], vrows) if vrows else \
        "<div class='note'>Nothing flagged.</div>"

    gal = []
    for v in viols[:max_evidence]:
        p = v.get("evidence")
        if not p or not os.path.exists(p):
            continue
        b = _b64_img(p, max_w=340)
        if not b:
            continue
        gal.append(f"<div class='ev'><img src='{b}' alt='evidence'>"
                   f"<div class='m'><b>{_esc(v['type'])}</b><br>ID {v['id']} · "
                   f"{_esc(v['class'])} · {_ts(v['timestamp'])} · "
                   f"{_esc(v['lane'])}</div></div>")
    gal_html = ("<h2>Evidence</h2><div class='gal'>" + "".join(gal) + "</div>") if gal else ""

    lane_rows = res.get("lane_rows") or []
    lkeys = [k for k in (lane_rows[0].keys() if lane_rows else []) if not k.startswith("_")]
    lrows = []
    for r in lane_rows:
        cells = []
        for k in lkeys:
            if k == "LOS":
                cells.append(f"<span class='pill' style='background:"
                             f"{r.get('_los_color', '#6b7280')}'>{_esc(r[k])}</span>")
            else:
                cells.append(_esc(r[k]))
        lrows.append(cells)
    lanes_html = _table(lkeys, lrows) if lrows else ""

    t_rows = [[tid, d["class"], d["lane"], d["frames"], d["dist_m"], d["avg_speed"], d["peak_speed"], d["max_offset_m"], "yes" if d["counted"] else "no", ", ".join(d["flags"])] for tid, d in sorted((res.get("tracks") or {}).items())]
    tracks_html = _table(["Track", "Vehicle", "Lane", "Frames", "Distance (m)", "Avg (km/h)", "Peak (km/h)", "Max Offset (m)", "Counted", "Flags"], t_rows) if t_rows else "<div class='note'>No tracks analyzed.</div>"

    lc_rows = [[_ts(c["t"]), c["id"], c["cls"], lm.name(c["from"]) if lm else c["from"], lm.name(c["to"]) if lm else c["to"], c["n"], round(c["dur"], 2), round(c["speed"], 1), ", ".join(c["types"])] for c in sorted(res.get("lane_changes") or [], key=lambda c: c["t"])]
    lane_changes_html = _table(["Time", "Track", "Vehicle", "From", "To", "Lanes Crossed", "Duration (s)", "Speed", "Lines Crossed"], lc_rows) if lc_rows else "<div class='note'>No lane changes detected.</div>"

    try:
        desc = PerspectiveCalib.from_dict(pers).describe()
    except Exception:
        desc = {}
    krows = [[_esc(k), f"<span class='mono'>{_esc(v)}</span>"] for k, v in desc.items()]
    btypes = cal.get("boundary_types") or []
    bfill = cal.get("boundary_fill") or []
    brows = [[f"Line {i + 1}", _esc(t), f"{f:.0%}"]
             for i, (t, f) in enumerate(zip(btypes, bfill))]
    rep = cal.get("report") or {}
    steps = "".join(f"<div class='note'>{_esc(s)}</div>" for s in (rep.get("steps") or []))
    notes = "".join(f"<div class='note warn'>{_esc(s)}</div>" for s in (rep.get("notes") or []))
    diag = ""
    if os.path.exists(DIAG_IMAGE):
        b = _b64_img(DIAG_IMAGE, max_w=1100)
        if b:
            diag = f"<h2>Geometry preview</h2><img class='diag' src='{b}' alt='diagnostics'>"
    approx = str(pers.get("scale_src", "")).lower()
    warn = ""
    if "assum" in approx or "approx" in approx:
        warn = ("<div class='note warn'><b>Speeds are approximate.</b> No usable dashed-"
                "marking rhythm was found, so the focal length was assumed. Lane "
                "positions and lateral measurements are unaffected; only longitudinal "
                "distance and therefore speed carry this uncertainty.</div>")
    calib_html = (warn + diag + "<h2>Perspective model</h2>"
                  + _table(["Parameter", "Value"], krows)
                  + ("<h2>Boundary lines</h2>" + _table(["Line", "Type", "Paint fill"], brows)
                     if brows else "")
                  + "<h2>How this video was calibrated</h2>" + steps + notes)

    tabs = ["Overview", "Violations", "Lanes", "Tracks", "Lane Changes", "Calibration"]
    tab_html = "".join(
        "<div class='tab%s' onclick='tab(%d)'>%s</div>" % (" on" if i == 0 else "", i, t)
        for i, t in enumerate(tabs))
    html = f"""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{APP_NAME} {APP_VER} report</title><style>{_CSS}</style></head><body class="dark"><div class="wrap">
<button class="theme-btn" onclick="document.body.classList.toggle('dark')">🌓 Toggle Light/Dark Mode</button>
<h1>{APP_NAME} <span style="color:#38bdf8">{APP_VER}</span></h1>
<div class="sub">{_esc(os.path.basename(res.get('video', '') or 'clip'))} &middot;
{res.get('width')}x{res.get('height')} &middot; {_ts(res.get('duration', 0))} &middot;
analysed in {res.get('elapsed', 0):.0f}s on {_esc(res.get('device', ''))} &middot;
{_esc(res.get('created', ''))}</div>
<div class="tabs">{tab_html}</div>
<div class="pane on">{overview}</div>
<div class="pane">{vtab}{gal_html}</div>
<div class="pane">{lanes_html}</div>
<div class="pane">{tracks_html}</div>
<div class="pane">{lane_changes_html}</div>
<div class="pane">{calib_html}</div>
</div><script>{_JS}</script></body></html>"""
    return html

# ==============================================================================
#  SECTION 11 - MANUAL CALIBRATION (4 clicks per lane)
# ==============================================================================
def _vp_from_polylines(polylines, frame_shape):
    h, w = frame_shape[:2]
    segs = []
    for pl in polylines:
        P = np.asarray(pl, np.float64)
        if P.shape[0] < 2:
            continue
        (x1, y1), (x2, y2) = P[0], P[-1]
        if abs(y2 - y1) < 2:
            continue
        segs.append((float(x1), float(y1), float(x2), float(y2),
                     float(math.hypot(x2 - x1, y2 - y1))))
    if len(segs) < 2:
        return (w * 0.5, h * 0.25), 0.0
    top = min(min(s[1], s[3]) for s in segs)
    vp, conf, _ = ransac_vanishing_point(segs, frame_shape, top, iters=400)
    if vp is not None:
        return vp, max(conf, 0.35)
    pts = []
    for i in range(len(segs)):
        for j in range(i + 1, len(segs)):
            a = _line_from_seg(*segs[i][:4])
            b = _line_from_seg(*segs[j][:4])
            p = np.cross(a, b)
            if abs(p[2]) < 1e-9:
                continue
            x, y = p[0] / p[2], p[1] / p[2]
            if -3 * w < x < 4 * w and -4 * h < y < top + 0.1 * h:
                pts.append((x, y))
    if not pts:
        return (w * 0.5, min(top - 0.05 * h, h * 0.25)), 0.15
    P = np.asarray(pts)
    return (float(np.median(P[:, 0])), float(np.median(P[:, 1]))), 0.3

def manual_calibration(quads, frame_bgr, marking_std="Auto (measure & guess)",
                       lane_w_m=DEFAULT_LANE_WIDTH_M, video_sig="manual"):
    if not quads:
        return None, "Draw at least one lane first."
    h, w = frame_bgr.shape[:2]
    order = sorted(range(len(quads)),
                   key=lambda i: float(np.mean([p[0] for p in quads[i]])))
    lanes, edges = {}, []
    for n, i in enumerate(order):
        P = np.asarray(quads[i], np.float64)
        L, R = _poly_edges({"polygon": P.tolist()})
        nm = f"Lane {n + 1}"
        lanes[nm] = {
            "polygon": [[int(round(p[0])), int(round(p[1]))] for p in P],
            "left": [[float(p[0]), float(p[1])] for p in L],
            "right": [[float(p[0]), float(p[1])] for p in R],
            "flow": 0, "support": 0.0, "kind": "travel",
            "left_type": "unknown", "right_type": "unknown",
        }
        edges += [L, R]

    vp, vconf = _vp_from_polylines(edges, frame_bgr.shape)

    names = list(lanes.keys())
    left_rows = [_rows_from_polyline(lanes[nm]["left"], h, vp) for nm in names]
    right_rows = [_rows_from_polyline(lanes[nm]["right"], h, vp) for nm in names]
    bnd_rows = [left_rows[0]] + right_rows
    pairs = list(zip(left_rows, right_rows))

    a, r2, _y_free = fit_width_model(bnd_rows, vp[1], h, pairs=pairs)
    if a is None or a <= 1e-6:
        y_b = int(_clamp(max(p[1] for L in lanes.values()
                             for p in L["polygon"]) - 1, 1, h - 1))
        widths = [right_rows[i][y_b] - left_rows[i][y_b] for i in range(len(names))
                  if np.isfinite(left_rows[i][y_b]) and np.isfinite(right_rows[i][y_b])
                  and right_rows[i][y_b] > left_rows[i][y_b]]
        if not widths:
            return None, "Could not measure lane width from the drawn shape."
        a = float(np.median(widths)) / max(1.0, (y_b - vp[1]))
        r2 = 0.0

    band = np.zeros((h, w), np.uint8)
    for L in lanes.values():
        cv2.fillPoly(band, [np.asarray(L["polygon"], np.int32)], 255)
    band = cv2.dilate(band, cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (max(3, w // 160), max(3, w // 160))))
    try:
        paint = paint_mask(frame_bgr, band)
    except Exception:
        paint = np.zeros((h, w), np.uint8)

    ys = [p[1] for L in lanes.values() for p in L["polygon"]]
    y_near, y_far = float(max(ys)), float(min(ys))
    std_m = MARKING_STANDARDS.get(marking_std, (None, ""))[0]
    k, src = None, ""
    if std_m and std_m > 0:
        k, strength, nlines = dash_period_scale(paint, bnd_rows, vp[1], y_near,
                                                y_far, std_m, frame_bgr.shape)
        if k:
            src = (f"dashed markings, {std_m:.1f} m period over {nlines} line(s), "
                   f"rhythm {strength:.2f}")
    elif std_m is None:
        k, strength, nlines = dash_period_scale(paint, bnd_rows, vp[1], y_near,
                                                y_far, 9.0, frame_bgr.shape)
        if k:
            src = (f"dashed markings, assumed 9 m period over {nlines} line(s), "
                   f"rhythm {strength:.2f}")
    if not k:
        src = "assumed focal length - speeds are APPROXIMATE"

    calib = PerspectiveCalib(vp, vp[1], a, float(lane_w_m), k=k, f=None,
                             scale_src=src, conf=vconf, frame_w=w, frame_h=h,
                             width_fit_r2=r2)
    try:
        btypes = classify_boundaries(paint, bnd_rows, vp[1], y_near, y_far,
                                     frame_bgr.shape)
    except Exception:
        btypes = [{"type": "unknown", "fill": 0.0, "rhythm": 0.0}
                  for _ in bnd_rows]
    for i, nm in enumerate(names):
        lanes[nm]["left_type"] = btypes[i]["type"] if i < len(btypes) else "unknown"
        lanes[nm]["right_type"] = btypes[i + 1]["type"] if i + 1 < len(btypes) else "unknown"

    rep = {"steps": [f"Manual mode: {len(lanes)} lane(s) drawn by hand.",
                     f"Vanishing point from the drawn edges: "
                     f"({vp[0]:.0f}, {vp[1]:.0f}), confidence {vconf:.0%}.",
                     f"Lane width model {a:.4f} px/row (R^2 {r2:.2f}).",
                     f"Longitudinal scale: {src}."],
           "notes": []}
    if r2 and r2 < 0.85:
        rep["notes"].append("The drawn edges are not very parallel (low R^2). "
                            "Re-draw with the lane sides following the markings "
                            "more closely for better speed accuracy.")
    if not k:
        rep["notes"].append("No dashed rhythm found on the drawn lines, so the "
                            "focal length was assumed. Lane positions and lateral "
                            "measurements are still exact; speeds are approximate.")
    return {
        "schema": CALIB_SCHEMA, "video_sig": video_sig,
        "frame_w": w, "frame_h": h, "analysis_scale": 1.0,
        "lanes": lanes, "perspective": calib.to_dict(),
        "engine": "Manual (hand-drawn lanes)", "marking_std": marking_std,
        "vp_source": "drawn lane edges", "vp_conf": vconf,
        "geometry_conf": vconf, "pitch_bev": None, "n_lanes": len(lanes),
        "boundary_types": [b["type"] for b in btypes],
        "boundary_fill": [round(b["fill"], 3) for b in btypes],
        "report": rep, "elapsed": 0.0,
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    }, None

def add_manual_lanes(existing_calib, quads, frame_bgr, marking_std="Auto (measure & guess)"):
    if not existing_calib:
        return None, "Auto-calibrate (or load a saved calibration) first."
    if not quads:
        return None, "Draw at least one lane first."

    h, w = frame_bgr.shape[:2]
    pers = existing_calib["perspective"]
    vp = (pers["vx"], pers["vy"])

    edge_pairs = []
    old_lanes_list = list(existing_calib.get("lanes", {}).values())
    for L in old_lanes_list:
        edge_pairs.append((np.asarray(L["left"], np.float64),
                           np.asarray(L["right"], np.float64)))
    for q in quads:
        P = np.asarray(q, np.float64)
        L, R = _poly_edges({"polygon": P.tolist()})
        edge_pairs.append((L, R))

    edge_pairs.sort(key=lambda pr: float(np.mean(pr[0][:, 0])))

    boundary_rows = [_rows_from_polyline(edge_pairs[0][0], h, vp)] + \
                    [_rows_from_polyline(pr[1], h, vp) for pr in edge_pairs]

    band = np.zeros((h, w), np.uint8)
    for L, R in edge_pairs:
        poly = np.vstack([L, R[::-1]]).astype(np.int32)
        cv2.fillPoly(band, [poly], 255)
    band = cv2.dilate(band, cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (max(3, w // 160), max(3, w // 160))))
    try:
        paint = paint_mask(frame_bgr, band)
        ys_all = [p[1] for L, R in edge_pairs for p in np.vstack([L, R])]
        y_near, y_far = float(max(ys_all)), float(min(ys_all))
        btypes = classify_boundaries(paint, boundary_rows, vp[1], y_near, y_far,
                                     frame_bgr.shape)
    except Exception:
        btypes = [{"type": "unknown", "fill": 0.0, "rhythm": 0.0} for _ in boundary_rows]

    new_lanes = {}
    for i, _ in enumerate(edge_pairs):
        Lf = _full_span_polyline(boundary_rows[i], vp, h)
        Rf = _full_span_polyline(boundary_rows[i + 1], vp, h)
        if Lf.shape[0] < 2 or Rf.shape[0] < 2:
            continue
        poly = np.vstack([Lf, Rf[::-1]])
        nm = f"Lane {i + 1}"

        c_new = (Lf[-1][0] + Rf[-1][0]) / 2.0
        best_flow, best_sup, best_kind = 0, 0.0, "travel"
        for old_L in old_lanes_list:
            if len(old_L.get("polygon", [])) >= 4:
                c_old = np.mean([p[0] for p in old_L["polygon"][-2:]])
                if abs(c_new - c_old) < (Rf[-1][0] - Lf[-1][0]) * 0.5:
                    best_flow = old_L.get("flow", 0)
                    best_sup = old_L.get("support", 0.0)
                    best_kind = old_L.get("kind", "travel")
                    break

        new_lanes[nm] = {
            "polygon": [[int(round(p[0])), int(round(p[1]))] for p in poly],
            "left": Lf.tolist(), "right": Rf.tolist(),
            "left_type": btypes[i]["type"] if i < len(btypes) else "unknown",
            "right_type": btypes[i + 1]["type"] if i + 1 < len(btypes) else "unknown",
            "flow": best_flow, "support": best_sup, "kind": best_kind,
        }

    if not new_lanes:
        return None, "Could not merge the drawn lanes - try redrawing them."

    merged = dict(existing_calib)
    merged["lanes"] = new_lanes
    merged["n_lanes"] = len(new_lanes)
    merged["engine"] = str(existing_calib.get("engine", "")) + " + manual additions"
    merged["boundary_types"] = [b["type"] for b in btypes]
    merged["boundary_fill"] = [round(b["fill"], 3) for b in btypes]
    old_rep = dict(existing_calib.get("report") or {})
    new_notes = list(old_rep.get("notes") or [])
    new_notes.append(f"{len(quads)} lane(s) added by hand and merged with the "
                     f"automatic result ({len(new_lanes)} lanes total). Traffic "
                     f"flows preserved to ensure Wrong-Way detection stays accurate.")
    old_rep["notes"] = new_notes
    merged["report"] = old_rep
    return merged, None

# ==============================================================================
#  SECTION 12 - CALIBRATION CACHE
# ==============================================================================
def calib_path_for(sig):
    return os.path.join(CALIB_DIR, f"calib_{sig}.json")

def save_calibration(calib):
    if not calib:
        return None
    p = calib_path_for(calib.get("video_sig", "manual"))
    try:
        with open(p, "w") as f:
            json.dump({k: v for k, v in calib.items() if k != "lane_model"}, f)
        return p
    except Exception:
        return None

def load_calibration(sig):
    p = calib_path_for(sig)
    if not os.path.exists(p):
        return None
    try:
        with open(p) as f:
            d = json.load(f)
        if int(d.get("schema", 0)) != CALIB_SCHEMA:
            return None
        return d
    except Exception:
        return None

def calib_summary_md(calib):
    if not calib:
        return "_No calibration yet._"
    pers = calib.get("perspective") or {}
    rep = calib.get("report") or {}
    lines = [
        f"### {calib.get('n_lanes', 0)} lanes  ·  {calib.get('engine', '')}",
        f"**Vanishing point:** {calib.get('vp_source', '?')} "
        f"(confidence {100 * float(calib.get('vp_conf', 0)):.0f}%)  ",
        f"**Geometry confidence:** {100 * float(calib.get('geometry_conf', 0)):.0f}%  ",
        f"**Longitudinal scale:** {pers.get('scale_src', '?')}  ",
        f"**Boundary lines:** " + ", ".join(
            f"{t}" for t in (calib.get("boundary_types") or [])) + "  ",
        "",
    ]
    if rep.get("steps"):
        lines.append("**What the calibrator did**")
        lines += [f"- {s}" for s in rep["steps"]]
    if rep.get("notes"):
        lines.append("")
        lines.append("**Warnings**")
        lines += [f"- {s}" for s in rep["notes"]]
    return "\n".join(lines)

# ==============================================================================
#  SECTION 12b - MEASURABLE METRICS
METRICS_LOG = os.path.join(SAVE_DIR, "metrics_report.txt")

def _out(line=""):
    try:
        sys.__stdout__.write(str(line) + "\n")
        sys.__stdout__.flush()
    except Exception:
        print(line)
    try:
        with open(METRICS_LOG, "a", encoding="utf-8") as f:
            f.write(str(line) + "\n")
    except Exception:
        pass

def _row(rows, k, v, where):
    rows.append((str(k), str(v), str(where)))

def _out_table(title, rows):
    _out(f"\n--- {title} " + "-" * max(0, 60 - len(title)))
    w1 = max([len(r[0]) for r in rows] + [6])
    w2 = max([len(r[1]) for r in rows] + [5])
    _out(f"  {'Metric':<{w1}} | {'Value':<{w2}} | Where")
    _out(f"  {'-'*w1}-+-{'-'*w2}-+-{'-'*22}")
    for k, v, where in rows:
        _out(f"  {k:<{w1}} | {v:<{w2}} | {where}")

def print_calibration_report(calib):
    """Call right after calibrate_video() / auto-calibration finishes."""
    if not calib:
        _out("No calibration to report on yet.")
        return
    _out(f"\n{'#'*72}\n# CALIBRATION METRICS - {time.strftime('%Y-%m-%d %H:%M:%S')}\n{'#'*72}")
    pers = calib.get("perspective") or {}
    rep = calib.get("report") or {}
    fills = calib.get("boundary_fill") or []
    avg_fill = (100.0 * sum(fills) / len(fills)) if fills else 0.0
    rows = []
    _row(rows, "Lanes detected", calib.get("n_lanes", 0), "Lane calibration")
    _row(rows, "Calibration engine", calib.get("engine", "-"), "Lane calibration")
    _row(rows, "Vanishing-point source", calib.get("vp_source", "-"), "RANSAC / trajectory fusion")
    _row(rows, "Vanishing-point confidence",
         f"{100*float(calib.get('vp_conf', 0)):.1f}%", "RANSAC / trajectory fusion")
    _row(rows, "Lane-geometry confidence",
         f"{100*float(calib.get('geometry_conf', 0)):.1f}%", "Marking + trajectory fusion")
    _row(rows, "Lane-width parallelism fit (R^2)",
         f"{float(pers.get('width_fit_r2', 0)):.3f}", "Lane-width model")
    _row(rows, "Longitudinal-scale confidence",
         f"{100*float(pers.get('conf', 0)):.1f}%", f"Scale src: {pers.get('scale_src','-')}")
    _row(rows, "Lane-marking pixel coverage",
         f"{avg_fill:.1f}% avg boundary fill", "Paint mask")
    _row(rows, "Calibration time", f"{calib.get('elapsed', 0):.1f}s", "Auto-calibration pass")
    _row(rows, "Data-quality warnings raised", len(rep.get("warnings") or []), "Auto-calibration pass")
    _out_table("LANE CALIBRATION METRICS", rows)
    if rep.get("warnings"):
        _out("  Warnings:")
        for s in rep["warnings"]:
            _out(f"    - {s}")
    _out("-" * 72)
    _out(f"(also saved to {METRICS_LOG})")

def print_analysis_report(res):
    """Call right after analyze_video() finishes (this is the full per-run report)."""
    if not res:
        _out("No analysis result to report on yet.")
        return
    _out(f"\n{'#'*72}\n# ANALYSIS METRICS - {time.strftime('%Y-%m-%d %H:%M:%S')}\n{'#'*72}")
    calib = res.get("calibration") or {}
    pers = calib.get("perspective") or {}
    tracks = res.get("tracks") or {}
    viol = res.get("violations") or []
    per_frame = res.get("per_frame") or []
    n_tracks = len(tracks)
    frames_done = res.get("frames_processed", 0) or 0
    elapsed = res.get("elapsed", 0.0) or 1e-9
    duration = res.get("duration", 0.0) or 0.0
    stride = res.get("stride", 1) or 1
    fps = res.get("fps", 0.0) or 1e-6
    proc_fps = frames_done / elapsed if elapsed > 0 else 0.0
    rtf = duration / elapsed if elapsed > 0 else 0.0

    conf_by_type = defaultdict(list)
    evid_hits = 0
    for v in viol:
        conf_by_type[v.get("type", "?")].append(float(v.get("confidence", 0.0)))
        if v.get("evidence"):
            evid_hits += 1
    evid_rate = (100.0 * evid_hits / len(viol)) if viol else 0.0

    durations_sec = [(t.get("frames", 0) * stride) / fps for t in tracks.values()]
    avg_dur = sum(durations_sec) / len(durations_sec) if durations_sec else 0.0
    stable_tracks = sum(1 for d in durations_sec if d >= 1.0)
    stable_rate = (100.0 * stable_tracks / n_tracks) if n_tracks else 0.0

    live_counts = [n for _, n, _ in per_frame] if per_frame else []
    avg_live = sum(live_counts) / len(live_counts) if live_counts else 0.0
    peak_live = max(live_counts) if live_counts else 0

    model_info = res.get("model") or {}

    rows = []
    _row(rows, "Frames processed", f"{frames_done} (every {stride} frames)", "Runtime, this run")
    _row(rows, "Video duration", f"{duration:.1f}s", "Runtime, this run")
    _row(rows, "Processing time (wall clock)", f"{elapsed:.1f}s", f"Runtime, {res.get('device','-')}")
    _row(rows, "Processing speed", f"{proc_fps:.2f} FPS", f"Runtime, {res.get('device','-')}")
    _row(rows, "Real-time factor", f"{rtf:.2f}x", "Runtime (video-seconds analysed per wall second)")
    _row(rows, "Model", model_info.get("path", "-"), "YOLO detection + ByteTrack")
    _row(rows, "Unique vehicles tracked", n_tracks, "ByteTrack, this run")
    _row(rows, "Vehicles counted (line crossing)", res.get("total", 0), "Counting line, this run")
    _row(rows, "Total PCU", res.get("pcu_total", 0), "PCU weighting, this run")
    _row(rows, "Avg. concurrent vehicles / frame", f"{avg_live:.1f} (peak {peak_live})", "ByteTrack, this run")
    _row(rows, "Avg. track duration", f"{avg_dur:.1f}s", "ByteTrack, this run")
    _row(rows, "Stable-track rate (>=1s tracked)", f"{stable_rate:.1f}% ({stable_tracks}/{n_tracks})", "ByteTrack, this run")
    _row(rows, "Mean traffic speed", f"{res.get('avg_speed','-')} km/h", "Metric speed calibration")
    _row(rows, "Level of Service", res.get("los", "-"), "Speed vs. speed-limit ratio")
    _row(rows, "Lanes detected", calib.get("n_lanes", 0), "Lane calibration")
    _row(rows, "Vanishing-point confidence", f"{100*float(calib.get('vp_conf', 0)):.1f}%", "Lane calibration")
    _row(rows, "Lane-geometry confidence", f"{100*float(calib.get('geometry_conf', 0)):.1f}%", "Lane calibration")
    _row(rows, "Lane-width parallelism fit (R^2)", f"{float(pers.get('width_fit_r2', 0)):.3f}", "Lane calibration")
    _row(rows, "Violations logged (total)", len(viol), "Violation engine, this run")
    _row(rows, "Vehicles flagged", f"{res.get('violation_rate', 0)}% of {n_tracks} tracked", "Violation engine, this run")
    _row(rows, "Evidence capture rate", f"{evid_rate:.1f}% ({evid_hits}/{len(viol)})" if viol else "n/a", "Violation engine, this run")
    _out_table("SYSTEM PERFORMANCE & DETECTION METRICS - THIS RUN", rows)

    if viol:
        vrows = []
        for t, n in sorted((res.get("violations_by_type") or {}).items(), key=lambda kv: -kv[1]):
            cs = conf_by_type.get(t, [])
            avgc = (100.0 * sum(cs) / len(cs)) if cs else 0.0
            _row(vrows, t, f"{n} events, avg conf {avgc:.0f}%", "Violation engine")
        _out_table("VIOLATIONS BY TYPE", vrows)

        srows = []
        for s, n in sorted((res.get("violations_by_severity") or {}).items(),
                            key=lambda kv: SEV_ORDER.get(kv[0], 9)):
            _row(srows, SEV_LABELS.get(s, s), n, "Violation engine")
        _out_table("VIOLATIONS BY SEVERITY", srows)

    if res.get("lane_rows"):
        _out(f"\n--- PER-LANE TRAFFIC IMPACT " + "-" * 43)
        hdr = f"  {'Lane':<10}{'Type':<10}{'Veh':>5}{'PCU/h':>9}{'AvgSpd':>9}{'Occ%':>7}  LOS"
        _out(hdr)
        for r in res["lane_rows"]:
            _out(f"  {r['Lane']:<10}{r['Type']:<10}{r['Vehicles']:>5}{r['PCU/h']:>9}"
                 f"{str(r['Avg speed (km/h)']):>9}{r['Occupancy (%)']:>7}  {r['LOS']}")
    _out("=" * 72)
    _out(f"(also saved to {METRICS_LOG})\n")

# ==============================================================================
#  SECTION 13 - USER INTERFACE
# ==============================================================================
S = {"video": None, "meta": None, "sig": None, "base": None,
     "calib": None, "clicks": [], "quads": [], "res": None}

HELP_MD = """
**How it works.** Load a clip, press *Auto-calibrate*, then *Run analysis*.
Calibration studies the whole video first: it builds a vehicle-free median
background to see the painted markings clearly, extracts white and yellow paint,
finds the vanishing point, warps the road to a bird's-eye view to recover the
lane spacing, and separately mines a short tracking prescan for the paths traffic
actually takes. Painted lanes and driven lanes are then cross-checked against each
other, so faded, patched or completely unmarked roads still come out with a
correct lane layout. Nothing here is tuned to one camera.

Missed a lane? Open *Draw lanes by hand* and click 4 corners for it, then press
*Merge Manual Lanes with Auto-Calibrated Lanes* to merge it with what auto-calibration already
found, instead of replacing it.
"""

def _draw_clicks(img, quads, clicks):
    out = img.copy()
    for i, q in enumerate(quads):
        P = np.asarray(q, np.int32)
        cv2.polylines(out, [P], True, (255, 150, 0), 2, cv2.LINE_AA)
        ov = out.copy()
        cv2.fillPoly(ov, [P], (255, 200, 0))
        cv2.addWeighted(ov, 0.18, out, 0.82, 0, out)
        c = P.mean(axis=0).astype(int)
        _label(out, f"Lane {i + 1}", c[0] - 24, c[1], (255, 150, 0), 0.5)
    for j, (x, y) in enumerate(clicks):
        cv2.circle(out, (int(x), int(y)), 6, (0, 255, 255), -1)
        cv2.circle(out, (int(x), int(y)), 6, (0, 0, 0), 1)
        _label(out, str(j + 1), int(x) + 8, int(y), (0, 255, 255), 0.42)
    if clicks:
        _label(out, f"{len(clicks)}/4 corners - a lane is added on the 4th click",
               10, out.shape[0] - 12, (0, 200, 255), 0.5)
    return out

def _preview(calib):
    if not calib or S["base"] is None:
        return S["base"]
    try:
        pc = PerspectiveCalib.from_dict(calib["perspective"])
        img = S["base"].copy()
        h, w = img.shape[:2]
        if abs(pc.frame_w - w) > 2:
            pc = pc.rescaled(w / float(pc.frame_w))
        lanes = _scaled_lanes(calib["lanes"], w / float(calib.get("frame_w", w) or w))
        lm = LaneModel(lanes, pc, (h, w))
        img = lm.draw(img)
        cv2.circle(img, (int(pc.vx), int(_clamp(pc.vy, 0, h - 1))), 5, (0, 255, 255), -1)
        _label(img, "vanishing point", int(pc.vx) + 8,
               int(_clamp(pc.vy, 14, h - 4)), (0, 255, 255), 0.45)
        for i, bt in enumerate(calib.get("boundary_types") or []):
            if i >= len(lm.boundary_rows):
                break
            yy = int(h * 0.90)
            xx = lm.boundary_rows[i][yy]
            if np.isfinite(xx):
                col = (255, 255, 255) if bt == "solid" else \
                      ((160, 160, 160) if bt == "dashed" else (90, 90, 200))
                _label(img, bt[:6], int(xx) - 18, yy, col, 0.38)
        return img
    except Exception:
        return S["base"]

def ui_load(video, model_path):
    if not video:
        return ("Pick a video first.", None, None, "_No calibration yet._", None)
    path = video if isinstance(video, str) else getattr(video, "name", None)
    if not path or not os.path.exists(path):
        return ("Could not read that file.", None, None, "_No calibration yet._", None)
    try:
        if os.path.abspath(path) != os.path.abspath(VIDEO_CACHE):
            shutil.copy(path, VIDEO_CACHE)
        path = VIDEO_CACHE
    except Exception:
        pass

    meta = video_meta(path)
    if not meta["ok"]:
        return ("That file could not be opened as a video.", None, None,
                "_No calibration yet._", None)
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(meta["frames"] * 0.35)))
    ok, frame = cap.read()
    if not ok:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ok, frame = cap.read()
    cap.release()
    if not ok:
        return ("Could not decode a frame from that video.", None, None,
                "_No calibration yet._", None)
    cv2.imwrite(BASE_FRAME, frame)

    S.update({"video": path, "meta": meta, "sig": _file_sig(path),
              "base": frame, "clicks": [], "quads": [], "calib": None, "res": None})
    if model_path:
        globals()["MODEL_PATH"] = model_path

    cached = load_calibration(S["sig"])
    cal_md = "_No calibration yet._"
    if cached:
        S["calib"] = cached
        cal_md = ("**A saved calibration was found for this exact video and "
                  "loaded.**\n\n" + calib_summary_md(cached))
    info = (f"**{os.path.basename(path)}**  \n"
            f"{meta['w']}x{meta['h']} · {meta['fps']:.1f} fps · "
            f"{_ts(meta['dur'])} · {meta['frames']} frames  \n"
            f"Compute: {device_label()}  \n"
            f"Signature: `{S['sig']}`")
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return (info, rgb, _draw_clicks(rgb, [], []),
            cal_md, cv2.cvtColor(_preview(cached), cv2.COLOR_BGR2RGB) if cached else rgb)

def ui_autocal(engine, std, progress=gr.Progress()):
    if not S["video"]:
        return "Load a video first.", None, None
    try:
        calib, msg = calibrate_video(
            S["video"], engine=engine, marking_std=std, lane_width_m=DEFAULT_LANE_WIDTH_M,
            n_lane_hint=0, det_conf=0.25, progress=progress)
    except Exception as e:
        err_msg = "Calibration failed: " + str(e) + "\n\n```text\n" + traceback.format_exc()[-1500:] + "\n```"
        return (err_msg, None, None)

    if not calib:
        return "Calibration failed: " + str(msg), None, None
    S["calib"] = calib
    save_calibration(calib)
    print_calibration_report(calib)
    diag = cv2.imread(DIAG_IMAGE) if os.path.exists(DIAG_IMAGE) else None
    return ("**" + str(msg) + "**\n\n" + calib_summary_md(calib),
            cv2.cvtColor(diag, cv2.COLOR_BGR2RGB) if diag is not None else None,
            cv2.cvtColor(_preview(calib), cv2.COLOR_BGR2RGB))

def ui_click(evt: gr.SelectData):
    if S["base"] is None:
        return None, "Load a video first."
    try:
        x, y = int(evt.index[0]), int(evt.index[1])
    except Exception:
        return None, "Could not read that click."
    S["clicks"].append((x, y))
    msg = f"Corner {len(S['clicks'])}/4 at ({x}, {y})."
    if len(S["clicks"]) == 4:
        pts = np.asarray(S["clicks"], np.float64)
        c = pts.mean(axis=0)
        ang = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])
        S["quads"].append(pts[np.argsort(ang)].tolist())
        S["clicks"] = []
        msg = f"Lane {len(S['quads'])} added. Click 4 more corners for the next lane."
    rgb = cv2.cvtColor(S["base"], cv2.COLOR_BGR2RGB)
    return _draw_clicks(rgb, S["quads"], S["clicks"]), msg

def ui_undo_point():
    if S["clicks"]:
        S["clicks"].pop()
    rgb = cv2.cvtColor(S["base"], cv2.COLOR_BGR2RGB) if S["base"] is not None else None
    return (_draw_clicks(rgb, S["quads"], S["clicks"]) if rgb is not None else None,
            "Removed the last corner.")

def ui_undo_lane():
    if S["quads"]:
        S["quads"].pop()
    rgb = cv2.cvtColor(S["base"], cv2.COLOR_BGR2RGB) if S["base"] is not None else None
    return (_draw_clicks(rgb, S["quads"], S["clicks"]) if rgb is not None else None,
            f"{len(S['quads'])} lane(s) remaining.")

def ui_clear_lanes():
    S["clicks"], S["quads"] = [], []
    rgb = cv2.cvtColor(S["base"], cv2.COLOR_BGR2RGB) if S["base"] is not None else None
    return (_draw_clicks(rgb, [], []) if rgb is not None else None, "Cleared.")

def ui_use_manual(std):
    if S["base"] is None:
        return "Load a video first.", None, None
    calib, err = manual_calibration(S["quads"], S["base"], marking_std=std,
                                    lane_w_m=DEFAULT_LANE_WIDTH_M,
                                    video_sig=S["sig"] or "manual")
    if not calib:
        return f"{err}", None, None
    S["calib"] = calib
    save_calibration(calib)
    return ("**Using the hand-drawn lanes.**\n\n" + calib_summary_md(calib), None,
            cv2.cvtColor(_preview(calib), cv2.COLOR_BGR2RGB))

def ui_add_manual(std):
    if S["base"] is None:
        return "Load a video first.", None, None
    if not S["quads"]:
        return "Draw at least one lane first.", None, None
    merged, err = add_manual_lanes(S["calib"], S["quads"], S["base"],
                                   marking_std=std)
    if not merged:
        return err, None, None
    S["calib"] = merged
    S["quads"], S["clicks"] = [], []
    save_calibration(merged)
    return ("**Added the hand-drawn lane(s) to the existing calibration.**\n\n" +
            calib_summary_md(merged), None,
            cv2.cvtColor(_preview(merged), cv2.COLOR_BGR2RGB))

def ui_load_cached():
    if not S["sig"]:
        return "Load a video first.", None, None
    c = load_calibration(S["sig"])
    if not c:
        return "No saved calibration for this video yet.", None, None
    S["calib"] = c
    return ("**Loaded the saved calibration.**\n\n" + calib_summary_md(c), None,
            cv2.cvtColor(_preview(c), cv2.COLOR_BGR2RGB))

def ui_save_calib():
    if not S.get("calib"):
        return "No calibration exists to save."
    path = save_calibration(S["calib"])
    if path:
        return f"**Calibration successfully saved to:** `{os.path.basename(path)}`\n\n" + calib_summary_md(S["calib"])
    return "Failed to save calibration."

def ui_download_calib():
    if not S["sig"]:
        return None
    p = calib_path_for(S["sig"])
    if os.path.exists(p):
        return p
    return None

def ui_run(speed_limit, stop_sec, straddle_ov, straddle_sec, viol_list, progress=gr.Progress()):
    if not S["video"]:
        return "Load a video first.", None, None, None
    if not S["calib"]:
        return ("Calibrate the lanes first - press *Auto-calibrate* on step 2, or "
                "draw them by hand."), None, None, None
    opts = {
        "speed_limit": speed_limit, "stop_sec": stop_sec,
        "straddle_overlap": straddle_ov, "straddle_sec": straddle_sec,
        "enabled_violations": list(viol_list or []),
        "model_path": MODEL_PATH,
    }
    try:
        res, err = analyze_video(S["video"], S["calib"], opts, progress=progress)
    except Exception as e:
        err_msg = "Analysis failed: " + str(e) + "\n\n```text\n" + traceback.format_exc()[-1800:] + "\n```"
        return (err_msg, None, None, None)

    if not res:
        return "Analysis failed: " + str(err), None, None, None
    S["res"] = res
    print_analysis_report(res)

    xlsx, xerr = build_excel(res)
    html = build_dashboard(res)
    try:
        with open(os.path.join(SAVE_DIR, "dashboard.html"), "w", encoding="utf-8") as f:
            f.write(html)
    except Exception:
        pass

    vt = res.get("violations_by_type") or {}
    top = ", ".join(f"{k} ({v})" for k, v in
                    sorted(vt.items(), key=lambda kv: -kv[1])[:4]) or "none"
    md = (f"### {res['total']} vehicles · {res['pcu_total']} PCU · "
          f"{len(res['violations'])} violations\n"
          f"**Speed:** {res.get('avg_speed') or '--'} km/h mean · "
          f"LOS {res.get('los')}  \n"
          f"**Offending vehicles:** {res.get('violation_rate')}% of "
          f"{len(res.get('tracks') or {})} tracks  \n"
          f"**Most common:** {top}  \n"
          f"**Lanes:** {res['calibration'].get('n_lanes')} "
          f"({res['lane_model'].n_travel()} carrying traffic)  \n"
          f"**Processed** {res['frames_processed']} frames in "
          f"{res['elapsed']:.0f}s on {res['device']}"
          + (f"  \n\n*{xerr}*" if xerr else ""))
    return md, res.get("out_video"), html, xlsx

def build_ui():
    if not GRADIO_OK:
        raise RuntimeError("gradio is not installed - install it to use the UI.")

    # Custom CSS for the button click effect and overall smoothness
    custom_css = """
    button:active {
        background-color: rgba(255, 255, 255, 0.9) !important;
        color: #000000 !important;
        box-shadow: 0 0 15px rgba(255, 255, 255, 0.8) !important;
        transition: all 0.1s ease;
    }
    """

    with gr.Blocks(title=f"{APP_NAME} {APP_VER}",
                   theme=gr.themes.Soft(primary_hue="blue"), css=custom_css) as demo:

        with gr.Row():
            gr.Markdown(f"# {APP_NAME} `{APP_VER}`\n"
                        f"Advanced lane detection and lane-violation analytics. "
                        f"Works on any fixed traffic camera - painted lanes, faded "
                        f"lanes, or no lane markings at all.")
            theme_btn = gr.Button("🌓 Toggle Light/Dark Mode", scale=0)

        theme_btn.click(None, None, None, js="() => { document.body.classList.toggle('dark'); }")

        with gr.Accordion("What this does", open=False):
            gr.Markdown(HELP_MD)

        with gr.Tab("1 · Video"):
            with gr.Row():
                with gr.Column(scale=1):
                    v_in = gr.Video(label="Traffic video")
                    mp_in = gr.Textbox(label="YOLO weights (optional)",
                                       value=MODEL_PATH,
                                       info="Leave as-is to use your trained model; "
                                            "falls back to COCO yolov10n if missing.")
                    load_btn = gr.Button("Load video", variant="primary")
                with gr.Column(scale=1):
                    info_md = gr.Markdown("_No video loaded._")
                    base_img = gr.Image(label="Reference frame", height=330)

        with gr.Tab("2 · Lanes"):
            gr.Markdown("Automatic calibration is the recommended path. Drawing by "
                        "hand adds lanes it missed, or replaces everything for "
                        "unusual scenes.")
            with gr.Row():
                eng_in = gr.Dropdown(ENGINE_MODES, value=ENGINE_MODES[0],
                                     label="Lane engine")
                std_in = gr.Dropdown(list(MARKING_STANDARDS.keys()),
                                     value=list(MARKING_STANDARDS.keys())[0],
                                     label="Marking standard",
                                     info="Sets the real-world dash period used "
                                          "to pin distance and speed.")
            with gr.Row():
                cal_btn = gr.Button("Auto-calibrate", variant="primary")
                cached_btn = gr.Button("Load saved calibration")
                save_calib_btn = gr.Button("Save Calibration")
                dl_calib_btn = gr.Button("Download Calibration")
            with gr.Row():
                cal_md = gr.Markdown("_No calibration yet._")
                calib_file = gr.File(label="Saved Calibration JSON", visible=True)
            with gr.Row():
                prev_img = gr.Image(label="Detected lanes", height=380)
                diag_img = gr.Image(label="Calibration diagnostics", height=380)

            with gr.Accordion("Draw lanes by hand (4 corners per lane)", open=False):
                draw_img = gr.Image(label="Click 4 corners per lane", height=430,
                                    interactive=False)
                draw_md = gr.Markdown("_Load a video, then click._")
                with gr.Row():
                    gr.Button("Undo corner").click(
                        ui_undo_point, outputs=[draw_img, draw_md])
                    gr.Button("Undo lane").click(
                        ui_undo_lane, outputs=[draw_img, draw_md])
                    gr.Button("Clear all").click(
                        ui_clear_lanes, outputs=[draw_img, draw_md])
                    add_btn = gr.Button("Merge Manual Lanes with Auto-Calibrated Lanes",
                                        variant="primary")
                    use_btn = gr.Button("Replace with these lanes only")

        with gr.Tab("3 · Analyse"):
            with gr.Row():
                with gr.Column(scale=1):
                    sl_in = gr.Slider(10, 150, value=DEFAULT_SPEED_LIMIT, step=5,
                                      label="Speed limit (km/h)")
                    st_in = gr.Slider(2, 30, value=DEFAULT_STOP_SEC, step=1,
                                      label="Stopped for this long = parking (s)")
                    so_in = gr.Slider(0.10, 0.60, value=STRADDLE_OVERLAP, step=0.02,
                                      label="Straddle: fraction of the vehicle over "
                                            "the line")
                    ss_in = gr.Slider(0.5, 5.0, value=STRADDLE_SEC, step=0.1,
                                      label="Straddle: seconds before it counts")
                with gr.Column(scale=1):
                    vl_in = gr.CheckboxGroup(list(VIOLATIONS.keys()),
                                             value=[k for k in VIOLATIONS],
                                             label="Violations to detect")

            run_btn = gr.Button("Run analysis", variant="primary", size="lg")
            res_md = gr.Markdown("_Nothing analysed yet._")
            with gr.Row():
                out_vid = gr.Video(label="Annotated video")
                out_xls = gr.File(label="Excel report")
            out_html = gr.HTML()

        load_btn.click(ui_load, [v_in, mp_in],
                       [info_md, base_img, draw_img, cal_md, prev_img])
        cal_btn.click(ui_autocal, [eng_in, std_in],
                      [cal_md, diag_img, prev_img])
        cached_btn.click(ui_load_cached, None, [cal_md, diag_img, prev_img])
        save_calib_btn.click(ui_save_calib, None, cal_md)
        dl_calib_btn.click(ui_download_calib, None, calib_file)

        draw_img.select(ui_click, None, [draw_img, draw_md])
        add_btn.click(ui_add_manual, [std_in], [cal_md, diag_img, prev_img])
        use_btn.click(ui_use_manual, [std_in], [cal_md, diag_img, prev_img])
        run_btn.click(ui_run,
                      [sl_in, st_in, so_in, ss_in, vl_in],
                      [res_md, out_vid, out_html, out_xls])
    return demo

def main():
    print(f"{APP_NAME} {APP_VER}  |  device: {device_label()}  |  data: {SAVE_DIR}")
    demo = build_ui()
    demo.queue().launch(share=True, debug=False, inbrowser=False, inline=False)

if __name__ == "__main__":
    main()

Connecting to Google Drive...
Mounted at /content/drive
Lane Guardian V30 PRO  |  device: GPU (Tesla T4)  |  data: /content/drive/MyDrive/Lane_Guardian_Data
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3056fc54a910167442.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
